# 🔬 Demo End-to-End: XAI + AI Agent Pipeline
## Multimodal Restaurant Review Scoring — CrossAttentionFusion (PhoBERT × Swin-B)

**Nhóm 24 — SE365 | Trình bày: Demo cuối kỳ**

Pipeline đầy đủ:
1. **Dự đoán** (CrossAttentionFusion) → 5 điểm: Food / Price / Atmosphere / Service / Overall
2. **Grad-CAM** — vùng ảnh quan trọng cho Overall Satisfaction
3. **PhoBERT Attention** — attention gộp ở mức từ, không hiển thị mảnh subword
4. **Cross-Attention** — tương tác hai chiều Token → Patch và Patch → Token
5. **SHAP** — đóng góp text-origin / image-origin sau cross-attention
6. **LIME** — giải thích cục bộ cho đúng mẫu đang xem
7. **AI Agent** (GPT-4o) — chỉ chạy khi Colab Secret `OPENAI_API_KEY` tồn tại

> **3 mẫu thử:** Mẫu A (dự đoán chính xác), Mẫu B (lỗi/xung đột), Mẫu C (đa ảnh/phong phú bằng chứng)

---
## 0.2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive mounted at /content/drive')

---
## 0.3 · Clone repo & checkout branch `final_demo`

In [ ]:
# ── 0.3 · Clone repo & checkout branch `final_demo` ─────────────────────────
import os
import shlex
import shutil
import subprocess

REPO_URL     = 'https://github.com/lechihoang/SE365.git'
PROJECT_ROOT = '/content/SE365'
BRANCH       = 'final_demo'


def run_cmd(cmd, cwd=None):
    """Run a command without invoking a shell; raise with captured output."""
    if not isinstance(cmd, (list, tuple)) or not cmd:
        raise TypeError('cmd must be a non-empty list/tuple of arguments')
    printable = shlex.join([str(part) for part in cmd])
    print(f'$ {printable}')
    result = subprocess.run(
        [str(part) for part in cmd],
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(
            f'Command failed ({result.returncode}): {printable}\n'
            f'{result.stdout or ""}'
        )
    return result.stdout or ''


os.chdir('/content')
if os.path.exists(PROJECT_ROOT):
    if not os.path.isdir(PROJECT_ROOT):
        raise RuntimeError(f'PROJECT_ROOT exists but is not a directory: {PROJECT_ROOT}')
    print(f'Removing existing repo: {PROJECT_ROOT}')
    shutil.rmtree(PROJECT_ROOT)

run_cmd([
    'git', 'clone', '--branch', BRANCH, '--single-branch',
    REPO_URL, PROJECT_ROOT,
])

git_dir = os.path.join(PROJECT_ROOT, '.git')
if not os.path.isdir(git_dir):
    raise FileNotFoundError(f'Git repository was not created: {git_dir}')

current_branch = run_cmd(
    ['git', 'branch', '--show-current'], cwd=PROJECT_ROOT
).strip()
if current_branch != BRANCH:
    raise RuntimeError(f'Expected branch {BRANCH!r}, got {current_branch!r}')

os.chdir(PROJECT_ROOT)
print(f'✅ Working directory: {os.getcwd()}')
print(f'✅ Branch: {current_branch}')

---
## 0.4 · Install thư viện bổ sung (nếu thiếu)

In [ ]:
# Install the repository requirements first, then demo-only dependencies.
import importlib.util
import sys

REQUIREMENTS_FILE = os.path.join(PROJECT_ROOT, 'requirements.txt')
if not os.path.isfile(REQUIREMENTS_FILE):
    raise FileNotFoundError(f'Missing requirements file: {REQUIREMENTS_FILE}')

run_cmd([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', REQUIREMENTS_FILE,
])

OPTIONAL_PACKAGES = {
    'shap': 'shap',
    'lime': 'lime',
    'seaborn': 'seaborn',
    'skimage': 'scikit-image',
    'dotenv': 'python-dotenv',
}
missing_packages = [
    package
    for module, package in OPTIONAL_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    run_cmd([
        sys.executable, '-m', 'pip', 'install', '-q',
        *missing_packages,
    ])

# The agent code uses the modern `OpenAI(...)` client API.
run_cmd([
    sys.executable, '-m', 'pip', 'install', '-q', 'openai>=1.0',
])

print('✅ Tất cả thư viện đã sẵn sàng.')

---
## 0.5 · Tải & giải nén dữ liệu (`data.zip`) từ Google Drive


In [ ]:
# ── Giải nén dữ liệu từ Google Drive, không dùng shell command ───────────────
import zipfile

DRIVE_DATA_ZIP = '/content/drive/MyDrive/SE365/data.zip'
DATA_DIR_LOCAL = os.path.join(PROJECT_ROOT, 'data')

if not os.path.isfile(DRIVE_DATA_ZIP):
    raise FileNotFoundError(
        f'Không tìm thấy data.zip trên Drive: {DRIVE_DATA_ZIP}'
    )

if os.path.isdir(DATA_DIR_LOCAL):
    print(f'Removing existing data directory: {DATA_DIR_LOCAL}')
    shutil.rmtree(DATA_DIR_LOCAL)

project_real = os.path.realpath(PROJECT_ROOT)
with zipfile.ZipFile(DRIVE_DATA_ZIP, 'r') as archive:
    members = archive.infolist()
    if not members:
        raise ValueError(f'Archive is empty: {DRIVE_DATA_ZIP}')

    # Reject path traversal before extracting.
    for member in members:
        destination = os.path.realpath(
            os.path.join(PROJECT_ROOT, member.filename)
        )
        if os.path.commonpath([project_real, destination]) != project_real:
            raise ValueError(
                f'Unsafe path in data archive: {member.filename!r}'
            )
    archive.extractall(PROJECT_ROOT)

expected_test_csv = os.path.join(DATA_DIR_LOCAL, 'text', 'test.csv')
expected_image_dir = os.path.join(DATA_DIR_LOCAL, 'image')
if not os.path.isfile(expected_test_csv):
    roots = sorted({
        member.filename.replace('\\', '/').split('/')[0]
        for member in members if member.filename
    })
    raise FileNotFoundError(
        'Giải nén hoàn tất nhưng không tìm thấy data/text/test.csv. '
        f'Các thư mục gốc trong archive: {roots}'
    )
if not os.path.isdir(expected_image_dir):
    raise FileNotFoundError(
        f'Giải nén hoàn tất nhưng thiếu thư mục ảnh: {expected_image_dir}'
    )

print(f'✅ Dữ liệu đã giải nén vào {DATA_DIR_LOCAL}')

---
## 0.6 · Imports


In [ ]:
import json
import traceback
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
warnings.filterwarnings('ignore')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from xai.config import (
    TARGET_NAMES, DISPLAY_NAMES, FACTOR_NAMES, LABEL_COLS,
    FUSED_DIM, CROSS_ATTN_HIDDEN_DIM,
    COLOR_SCHEMES, DEFAULT_DPI, THESIS_DPI,
    BEST_EXP_ID, DEFAULT_SEED,
    BEST_TEXT_MODEL, BEST_IMAGE_MODEL,
)
from xai.utils import (
    load_model, get_tokenizer, get_image_processor,
    load_single_sample, get_prediction,
    get_device, set_seed,
)
from xai.gradcam_explainer import (
    compute_gradcam_for_image, overlay_cam_on_image, find_target_layer,
)
from xai.attention_explainer import (
    extract_phobert_attention, aggregate_attention,
    cls_token_importance, merge_subword_attention,
    plot_cls_importance_bar,
    extract_cross_attention,
)
from xai.shap_explainer import (
    FusionHeadWrapper, extract_fused_embeddings,
    select_background, compute_shap_values, modality_contribution,
)
from xai.lime_explainer import (
    run_lime_image, run_lime_text,
    save_lime_image_explanation,
)
from xai.case_study import check_sample_artifacts

try:
    from agent import ExplanationAgent
    from agent.config import AgentConfig
    AGENT_AVAILABLE = True
except Exception as _agent_import_error:
    AGENT_AVAILABLE = False
    print(
        '[WARN] AI Agent import failed; XAI demo will continue without it: '
        f'{type(_agent_import_error).__name__}: {_agent_import_error}'
    )

set_seed(DEFAULT_SEED)
print('✅ Imports hoàn thành.')

---
## 0.7 · Cấu hình đường dẫn & thiết bị


In [ ]:
device = get_device()
print(f'Device: {device}')

DRIVE_ROOT = '/content/drive/MyDrive/SE365'
EXP_ID     = BEST_EXP_ID
EXP_DIR    = os.path.join(DRIVE_ROOT, 'experiments', EXP_ID)
XAI_DIR    = os.path.join(EXP_DIR, 'xai')
DEMO_OUT   = os.path.join(DRIVE_ROOT, 'demo_e2e')
AGENT_DIR  = os.path.join(DEMO_OUT, 'agent_reports')

DATA_DIR  = os.path.join(PROJECT_ROOT, 'data', 'text')
IMAGE_DIR = os.path.join(PROJECT_ROOT, 'data', 'image')
CSV_TEST  = os.path.join(DATA_DIR, 'test.csv')
CHECKPOINT_PATH = os.path.join(EXP_DIR, 'best_model_train_fusion.pth')

# Validate upstream inputs before creating output folders. Creating XAI_DIR
# first would accidentally create a missing EXP_DIR and hide the real error.
required_files = {
    'test CSV': CSV_TEST,
    'model checkpoint': CHECKPOINT_PATH,
}
required_dirs = {
    'experiment directory': EXP_DIR,
    'image cache': IMAGE_DIR,
}
missing_inputs = [
    f'{label}: {path}'
    for label, path in {**required_dirs, **required_files}.items()
    if not (os.path.isdir(path) if label in required_dirs else os.path.isfile(path))
]
if missing_inputs:
    raise FileNotFoundError(
        'Thiếu input bắt buộc cho demo:\n- ' + '\n- '.join(missing_inputs)
    )

for output_dir in [XAI_DIR, DEMO_OUT, AGENT_DIR]:
    os.makedirs(output_dir, exist_ok=True)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DRIVE_ROOT   : {DRIVE_ROOT}')
print(f'EXP_DIR      : {EXP_DIR}')
print(f'XAI_DIR      : {XAI_DIR}')
print(f'DEMO_OUT     : {DEMO_OUT}')
print(f'CSV_TEST     : {CSV_TEST}')
print(f'IMAGE_DIR    : {IMAGE_DIR}')

---
## 0.8 · Tải mô hình, tokenizer, image processor


In [ ]:
print('Đang tải mô hình CrossAttentionFusion...')
model, model_config = load_model(EXP_DIR, device)
model.eval()
print(f'✅ Mô hình: {type(model).__name__}')

text_model_name = model_config.get('text_model_name') or BEST_TEXT_MODEL
image_model_name = model_config.get('image_model_name') or BEST_IMAGE_MODEL
tokenizer = get_tokenizer(text_model_name)
image_processor = get_image_processor(image_model_name)

print(f'✅ Tokenizer: {type(tokenizer).__name__} ({text_model_name})')
print(f'✅ Image processor: {type(image_processor).__name__} ({image_model_name})')

target_layer = find_target_layer(model)
print(f'✅ Grad-CAM target layer: {type(target_layer).__name__}')

# load_model(..., xai_mode=True) already enables eager attention exactly once.
text_encoder = getattr(model.text_model, 'encoder', None)
attn_impl = getattr(
    getattr(text_encoder, 'config', None),
    '_attn_implementation',
    None,
)
print(f'✅ PhoBERT attention implementation: {attn_impl or "configured"}')

---
## 0.9 · Hàm tiện ích demo


In [ ]:
# ── run_safe: bao bọc mỗi bước XAI ─────────────────────────────────────────

def run_safe(fn, step_name='XAI step', fallback=None, **kwargs):
    try:
        return fn(**kwargs)
    except Exception as e:
        print(f'  [SKIP] {step_name}: {type(e).__name__}: {e}')
        if os.environ.get('XAI_TRACEBACK'):
            traceback.print_exc()
        return fallback

# ── Bảng dự đoán vs thực tế ─────────────────────────────────────────────────

def display_prediction_table(pred_result, sample_id=''):
    preds = pred_result['predictions']
    gt    = pred_result['ground_truth']
    errs  = pred_result['absolute_errors']
    rows  = []
    for name, disp in zip(TARGET_NAMES, DISPLAY_NAMES):
        rows.append({'Chỉ tiêu': disp,
                     'Thực tế' : f'{gt[name]:.1f}',
                     'Dự đoán' : f'{preds[name]:.1f}',
                     'AE'      : f'{errs[name]:.2f}'})
    df = pd.DataFrame(rows)
    mae = pred_result['mean_mae']
    print(f'\n{"="*50}')
    print(f'  {sample_id}  —  MAE = {mae:.3f}')
    print('='*50)
    print(df.to_string(index=False))
    print('='*50)
    return df

# ── Biểu đồ dự đoán ─────────────────────────────────────────────────────────

def plot_prediction_bars(pred_result, sample_id='', save_path=None):
    preds  = pred_result['predictions']
    gt     = pred_result['ground_truth']
    x      = range(len(TARGET_NAMES))
    p_vals = [preds[n] for n in TARGET_NAMES]
    g_vals = [gt[n]    for n in TARGET_NAMES]
    fig, ax = plt.subplots(figsize=(10, 4))
    w = 0.35
    b_gt   = ax.bar([i - w/2 for i in x], g_vals, w,
                    label='Thực tế', color=COLOR_SCHEMES['bar_gt'], alpha=0.85)
    b_pred = ax.bar([i + w/2 for i in x], p_vals, w,
                    label='Dự đoán', color=COLOR_SCHEMES['bar_pred'], alpha=0.85)
    for b in list(b_gt) + list(b_pred):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.1,
                f'{b.get_height():.1f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(DISPLAY_NAMES, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Điểm (1–10)', fontsize=10)
    ax.set_ylim(0, 12)
    ax.set_title(f'{sample_id} — Dự đoán vs Thực tế', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    fig.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
        print(f'[XAI] Đã lưu: {save_path}')
    plt.show()
    return fig

# ── Hiển thị ảnh review ──────────────────────────────────────────────────────

def show_review_images(sample, sample_id='', max_show=4):
    n = min(sample['num_real_images'], max_show)
    if n == 0:
        print('  (Không có ảnh thực)')
        return
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for i in range(n):
        img = sample['loaded_images'][i].convert('RGB').resize((224, 224))
        axes[i].imshow(np.array(img))
        axes[i].set_title(f'Ảnh {i+1}', fontsize=9)
        axes[i].axis('off')
    fig.suptitle(f'{sample_id} — {n} ảnh thực', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ── Dashboard tóm tắt ────────────────────────────────────────────────────────

def print_sample_summary(sample_id, pred_result, artifact_check):
    print(f'\n{chr(9473)*55}')
    print(f'  TÓM TẮT — {sample_id}')
    print(f'{chr(9473)*55}')
    for key, label in [('gradcam','Grad-CAM'),('attention','Attention'),
                        ('cross_attention','Cross-Attention'),('shap','SHAP'),('lime','LIME')]:
        status = '✅' if artifact_check.get(key) else '⚠️ '
        print(f'  {status} {label}')
    print(f"  Completeness: {artifact_check.get('completeness',0)*100:.0f}%")
    print(f"  MAE: {pred_result['mean_mae']:.3f}")
    for n, d in zip(TARGET_NAMES, DISPLAY_NAMES):
        e = pred_result['absolute_errors'][n]
        mark = '✅' if e <= 0.5 else ('⚠️ ' if e <= 1.5 else '❌')
        print(f'    {mark} {d}: AE={e:.2f}')
    print(f'{chr(9473)*55}\n')

# Bảng cross-method (điền dần)
CROSS_METHOD_ROWS = []

def add_cross_method_row(sample_id, pred_result, shap_contrib, lime_text_weights, case_type=''):
    top_words = ', '.join([w for w, _ in lime_text_weights[:3]]) if lime_text_weights else 'N/A'
    CROSS_METHOD_ROWS.append({
        'Mẫu'         : sample_id,
        'Loại'        : case_type,
        'MAE'         : f"{pred_result['mean_mae']:.3f}",
        'SHAP text'   : f"{shap_contrib.get('text_pct',0):.0f}%" if shap_contrib else 'N/A',
        'SHAP image'  : f"{shap_contrib.get('image_pct',0):.0f}%" if shap_contrib else 'N/A',
        'Top LIME'    : top_words,
    })

print('✅ Hàm tiện ích đã định nghĩa.')

# ── Shared robust helpers used by all three sample sections ──────────────────

def sample_to_batch(sample):
    """Adapt load_single_sample output to extract_fused_embeddings input."""
    labels = sample['factor_scores']
    if labels.dim() == 1:
        labels = labels.unsqueeze(0)
    return {
        'input_ids': sample['input_ids'],
        'attention_mask': sample['attention_mask'],
        'pixel_values': sample['pixel_values'],
        'num_images': sample['num_images'],
        'labels': labels,
    }


def _validate_background_tensor(value):
    if isinstance(value, dict):
        for key in ('background', 'background_fused', 'fused_embeddings'):
            if key in value:
                value = value[key]
                break
    if not isinstance(value, torch.Tensor):
        return None
    value = value.detach().cpu().float()
    if value.dim() != 2 or value.shape[1] != FUSED_DIM or value.shape[0] < 2:
        return None
    if not torch.isfinite(value).all():
        return None
    return value


def prepare_shap_background(candidate_indices, max_samples=8):
    """Load a valid cached SHAP baseline or build one from multiple samples."""
    cached_path = os.path.join(XAI_DIR, 'shap', 'raw', 'background_fused.pt')
    if os.path.isfile(cached_path):
        try:
            try:
                cached = torch.load(
                    cached_path, map_location='cpu', weights_only=True
                )
            except TypeError:
                cached = torch.load(cached_path, map_location='cpu')
            cached = _validate_background_tensor(cached)
            if cached is not None:
                print(
                    f'[SHAP] Reusing background: {cached_path} '
                    f'(shape={tuple(cached.shape)})'
                )
                return cached
            print(f'[SHAP] Cached background has an invalid shape: {cached_path}')
        except Exception as exc:
            print(
                f'[SHAP] Cannot load cached background '
                f'({type(exc).__name__}: {exc}); rebuilding.'
            )

    unique_indices = list(dict.fromkeys(int(i) for i in candidate_indices))
    if len(unique_indices) > max_samples:
        positions = np.linspace(
            0, len(unique_indices) - 1, num=max_samples, dtype=int
        )
        unique_indices = [unique_indices[pos] for pos in positions]

    batches = []
    failures = []
    for idx in unique_indices:
        try:
            sample = load_single_sample(
                csv_path=CSV_TEST,
                idx=idx,
                tokenizer=tokenizer,
                image_processor=image_processor,
                image_dir=IMAGE_DIR,
                device=device,
            )
            batches.append(sample_to_batch(sample))
        except Exception as exc:
            failures.append(f'idx={idx}: {type(exc).__name__}: {exc}')

    if len(batches) < 2:
        print(
            '[SHAP] Need at least 2 valid baseline samples; '
            f'found {len(batches)}. SHAP will be skipped safely.'
        )
        for message in failures[:3]:
            print(f'  - {message}')
        return None

    result = run_safe(
        extract_fused_embeddings,
        step_name='prepare_SHAP_background',
        fallback=(None, None, None),
        model=model,
        dataloader=batches,
        device=device,
        max_samples=len(batches),
    )
    fused = result[0] if result else None
    fused = _validate_background_tensor(fused)
    if fused is None:
        print('[SHAP] Background embedding extraction failed; SHAP will be skipped.')
        return None

    background, _ = select_background(
        fused,
        n_background=min(max_samples, fused.shape[0]),
        seed=DEFAULT_SEED,
    )
    print(f'[SHAP] Built multi-sample background: {tuple(background.shape)}')
    return background


def _cross_attention_word_groups(tokens):
    """Return readable word labels and source-token index groups."""
    special = {'<s>', '</s>', '<pad>', '<unk>', '<mask>'}
    mode = (
        'at_sign' if any('@@' in token for token in tokens)
        else 'g_prefix' if any(token.startswith('Ġ') for token in tokens)
        else 'none'
    )
    labels, groups = [], []

    if mode == 'at_sign':
        parts, indices = [], []
        for idx, token in enumerate(tokens):
            if token in special:
                continue
            parts.append(token[:-2] if token.endswith('@@') else token)
            indices.append(idx)
            if not token.endswith('@@'):
                labels.append(''.join(parts))
                groups.append(indices)
                parts, indices = [], []
        if indices:
            labels.append(''.join(parts))
            groups.append(indices)
    elif mode == 'g_prefix':
        current_label, indices = '', []
        for idx, token in enumerate(tokens):
            if token in special:
                continue
            if token.startswith('Ġ') and indices:
                labels.append(current_label)
                groups.append(indices)
                current_label, indices = token[1:], [idx]
            else:
                current_label += token[1:] if token.startswith('Ġ') else token
                indices.append(idx)
        if indices:
            labels.append(current_label)
            groups.append(indices)
    else:
        for idx, token in enumerate(tokens):
            if token not in special:
                labels.append(token)
                groups.append([idx])

    clean = [
        (label.strip() or f'word_{position}', group)
        for position, (label, group) in enumerate(zip(labels, groups))
    ]
    return [item[0] for item in clean], [item[1] for item in clean]


def merge_cross_attention_words(tokens, t2i_attn, i2t_attn):
    """Aggregate subwords for lecturer-facing T→P and P→T displays."""
    labels, groups = _cross_attention_word_groups(tokens)
    if not groups:
        raise ValueError('No non-special tokens available for cross-attention.')

    word_t2i = np.stack(
        [np.asarray(t2i_attn[group]).mean(axis=0) for group in groups],
        axis=0,
    )
    word_i2t = np.stack(
        [np.asarray(i2t_attn[:, group]).sum(axis=1) for group in groups],
        axis=1,
    )
    row_sums = word_i2t.sum(axis=1, keepdims=True)
    word_i2t = np.divide(
        word_i2t,
        row_sums,
        out=np.zeros_like(word_i2t),
        where=row_sums > 1e-12,
    )
    return labels, word_t2i, word_i2t


def save_lime_text_bar(raw_weights, save_path, title):
    """Save the standalone text artifact expected by check_sample_artifacts."""
    if not raw_weights:
        return None
    pairs = sorted(raw_weights, key=lambda item: abs(item[1]))[-10:]
    words = [word for word, _ in pairs]
    values = [float(weight) for _, weight in pairs]
    colors = ['#4CAF50' if value >= 0 else '#F44336' for value in values]
    fig, ax = plt.subplots(figsize=(8, max(4, 0.45 * len(words))))
    ax.barh(range(len(words)), values, color=colors)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('LIME weight (local sensitivity)')
    ax.set_title(title, fontweight='bold')
    fig.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return save_path

---
# 📋 PHẦN 1 — CHỌN MẪU DEMO

| Ký hiệu | Chiến lược | Tiêu chí |
|---|---|---|
| **Mẫu A** | Chính xác | MAE thấp nhất trong các hàng hợp lệ |
| **Mẫu B** | Lỗi/xung đột | MAE cao nhất, loại Mẫu A |
| **Mẫu C** | Bằng chứng đa phương thức | Nhiều ảnh nhất, MAE gần trung vị, loại A và B |

Mọi lựa chọn đều có fallback, kiểm tra dữ liệu số và đảm bảo ba chỉ số khác nhau.

In [ ]:
df_test = pd.read_csv(CSV_TEST)
print(f'Tổng mẫu test: {len(df_test)}')

required_dataset_columns = ['comment_clean', 'image_url', *LABEL_COLS]
missing_dataset_columns = [
    column for column in required_dataset_columns
    if column not in df_test.columns
]
if missing_dataset_columns:
    raise ValueError(
        f'Test CSV thiếu các cột bắt buộc: {missing_dataset_columns}'
    )
if df_test.empty:
    raise ValueError(f'Test CSV không có dòng dữ liệu: {CSV_TEST}')

SCAN_MAX = min(200, len(df_test))
print(f'Đang scan {SCAN_MAX} mẫu để tính MAE ...')

scan_results = []
for idx in range(SCAN_MAX):
    row_result = {
        'idx': idx,
        'mae': np.nan,
        'n_images': np.nan,
        'text_len': np.nan,
        'status': 'invalid',
        'error': '',
    }
    try:
        sample = load_single_sample(
            csv_path=CSV_TEST,
            idx=idx,
            tokenizer=tokenizer,
            image_processor=image_processor,
            image_dir=IMAGE_DIR,
            device=device,
        )
        prediction = get_prediction(model, sample)

        pred_values = np.asarray(
            prediction.get('predictions_array', []), dtype=float
        ).reshape(-1)
        true_values = np.asarray(
            prediction.get('ground_truth_array', []), dtype=float
        ).reshape(-1)
        mae = float(prediction.get('mean_mae', np.nan))

        if pred_values.size != len(TARGET_NAMES):
            raise ValueError(
                f'Expected {len(TARGET_NAMES)} predictions, got {pred_values.size}'
            )
        if true_values.size != len(TARGET_NAMES):
            raise ValueError(
                f'Expected {len(TARGET_NAMES)} labels, got {true_values.size}'
            )
        if not np.isfinite(pred_values).all():
            raise ValueError('Prediction contains NaN or infinity')
        if not np.isfinite(true_values).all():
            raise ValueError('Ground truth contains NaN or infinity')
        if not np.isfinite(mae):
            raise ValueError('MAE is NaN or infinity')

        row_result.update({
            'mae': mae,
            'n_images': int(sample.get('num_real_images', 0)),
            'text_len': len(str(sample.get('text', ''))),
            'status': 'valid',
        })
    except Exception as exc:
        row_result['error'] = f'{type(exc).__name__}: {exc}'
    scan_results.append(row_result)

REQUIRED_SCAN_COLUMNS = ['idx', 'mae', 'n_images', 'text_len']
df_scan = pd.DataFrame(
    scan_results,
    columns=[*REQUIRED_SCAN_COLUMNS, 'status', 'error'],
)

valid_mask = (
    df_scan['status'].eq('valid')
    & pd.to_numeric(df_scan['mae'], errors='coerce').notna()
)
df_scan_valid_preview = df_scan.loc[valid_mask].copy()

print(f'\nĐã scan          : {len(df_scan)} mẫu')
print(f'Hợp lệ           : {len(df_scan_valid_preview)} mẫu')
print(f'Không hợp lệ     : {len(df_scan) - len(df_scan_valid_preview)} mẫu')

if not df_scan_valid_preview.empty:
    mae_values = pd.to_numeric(
        df_scan_valid_preview['mae'], errors='coerce'
    )
    print(
        'MAE min/median/max: '
        f'{mae_values.min():.3f} / '
        f'{mae_values.median():.3f} / '
        f'{mae_values.max():.3f}'
    )
    image_distribution = (
        pd.to_numeric(
            df_scan_valid_preview['n_images'], errors='coerce'
        )
        .fillna(0)
        .astype(int)
        .value_counts()
        .sort_index()
        .rename_axis('n_images')
        .to_frame('sample_count')
    )
    print('\nPhân bố số ảnh:')
    print(image_distribution.to_string())
else:
    print(
        '\n[DIAGNOSTIC] df_scan không có mẫu hợp lệ. '
        'Kiểm tra đường dẫn dữ liệu, cache ảnh, tokenizer/processor, '
        'checkpoint và các cột nhãn.'
    )

failed_rows = df_scan.loc[~valid_mask, ['idx', 'error']]
if not failed_rows.empty:
    print('\nLỗi scan phổ biến (tối đa 10 dòng):')
    print(failed_rows.head(10).to_string(index=False))

In [ ]:
REQUIRED_SCAN_COLUMNS = ['idx', 'mae', 'n_images', 'text_len']


def pick_first(df, sort_cols, ascending=True, exclude=None, label='sample'):
    """Return the first valid idx after filtering/sorting, or None."""
    exclude = set(exclude or [])
    if df is None or df.empty:
        return None
    work = df.copy()
    work = work[~work['idx'].isin(exclude)]
    if work.empty:
        return None
    try:
        sorted_work = work.sort_values(
            sort_cols, ascending=ascending, kind='mergesort'
        )
    except (KeyError, ValueError) as exc:
        raise ValueError(
            f'Cannot sort candidates for {label}: {exc}'
        ) from exc
    if sorted_work.empty:
        return None
    return int(sorted_work.iloc[0]['idx'])


if df_scan is None or df_scan.empty:
    raise ValueError(
        'Không thể chọn mẫu: df_scan rỗng. '
        'Xem diagnostic ở cell scan để xác định bước upstream bị lỗi.'
    )

missing_cols = [
    column for column in REQUIRED_SCAN_COLUMNS
    if column not in df_scan.columns
]
if missing_cols:
    raise ValueError(f'df_scan thiếu các cột bắt buộc: {missing_cols}')

df_valid = df_scan.copy()
for column in REQUIRED_SCAN_COLUMNS:
    df_valid[column] = pd.to_numeric(df_valid[column], errors='coerce')

# idx must be finite, integer-valued, and point to an existing test row.
idx_is_integer = (
    df_valid['idx'].notna()
    & np.isfinite(df_valid['idx'])
    & np.isclose(df_valid['idx'] % 1, 0)
)
mae_is_valid = df_valid['mae'].notna() & np.isfinite(df_valid['mae'])
df_valid = df_valid.loc[idx_is_integer & mae_is_valid].copy()
df_valid['idx'] = df_valid['idx'].astype(int)
df_valid = df_valid[
    df_valid['idx'].between(0, len(df_test) - 1, inclusive='both')
]
df_valid['n_images'] = (
    df_valid['n_images'].fillna(0).clip(lower=0).astype(int)
)
df_valid['text_len'] = (
    df_valid['text_len'].fillna(0).clip(lower=0).astype(int)
)
df_valid = (
    df_valid
    .drop_duplicates(subset=['idx'], keep='first')
    .reset_index(drop=True)
)

if df_valid.empty:
    raise ValueError(
        'Không có candidate hợp lệ: df_scan không có dòng với idx nguyên '
        'hợp lệ và MAE hữu hạn. Xem bảng lỗi scan ở cell trước.'
    )
if len(df_valid) < 3:
    raise ValueError(
        'Cần ít nhất 3 mẫu hợp lệ và khác nhau cho demo, '
        f'nhưng chỉ tìm thấy {len(df_valid)}.'
    )

# A — globally lowest MAE. No quantile filter is required.
idx_a = pick_first(
    df_valid,
    sort_cols=['mae', 'n_images', 'text_len', 'idx'],
    ascending=[True, False, False, True],
    label='A_accurate',
)

# B — globally highest MAE after excluding A.
idx_b = pick_first(
    df_valid,
    sort_cols=['mae', 'n_images', 'text_len', 'idx'],
    ascending=[False, False, False, True],
    exclude=[idx_a],
    label='B_error',
)

# C — highest image count, then MAE nearest the remaining-set median.
remaining_c = df_valid[
    ~df_valid['idx'].isin([idx_a, idx_b])
].copy()
if remaining_c.empty:
    raise ValueError(
        'Không thể chọn Mẫu C: không còn candidate sau khi chọn A và B.'
    )
remaining_c['_mae_dist_to_median'] = (
    remaining_c['mae'] - remaining_c['mae'].median()
).abs()

preferred_c = remaining_c[remaining_c['n_images'] > 0].copy()
idx_c = pick_first(
    preferred_c,
    sort_cols=[
        'n_images', '_mae_dist_to_median', 'text_len', 'idx'
    ],
    ascending=[False, True, False, True],
    exclude=[idx_a, idx_b],
    label='C_multimodal_preferred',
)
selection_strategy_c = 'nhiều ảnh nhất + MAE gần trung vị'

if idx_c is None:
    idx_c = pick_first(
        remaining_c,
        sort_cols=['n_images', 'text_len', 'mae', 'idx'],
        ascending=[False, False, True, True],
        exclude=[idx_a, idx_b],
        label='C_image_text_fallback',
    )
    selection_strategy_c = 'fallback: nhiều ảnh, rồi review dài'

if idx_c is None:
    idx_c = pick_first(
        df_valid,
        sort_cols=['text_len', 'idx'],
        ascending=[False, True],
        exclude=[idx_a, idx_b],
        label='C_any_remaining',
    )
    selection_strategy_c = 'fallback: bất kỳ mẫu còn lại'

if None in (idx_a, idx_b, idx_c):
    raise ValueError(
        f'Không chọn đủ 3 mẫu: A={idx_a}, B={idx_b}, C={idx_c}'
    )
if len({idx_a, idx_b, idx_c}) != 3:
    raise RuntimeError(
        f'Invariant violated: A, B, C phải khác nhau, got '
        f'{idx_a}, {idx_b}, {idx_c}.'
    )

SAMPLE_INDICES = {'A': idx_a, 'B': idx_b, 'C': idx_c}
SAMPLE_IDS = {
    key: f'sample_{idx:04d}'
    for key, idx in SAMPLE_INDICES.items()
}
CASE_TYPES = {'A': 'accurate', 'B': 'error', 'C': 'multimodal'}

selected_rows = (
    df_valid[df_valid['idx'].isin(SAMPLE_INDICES.values())]
    .set_index('idx')
)
summary_rows = []
for key, strategy in [
    ('A', 'MAE thấp nhất'),
    ('B', 'MAE cao nhất, loại A'),
    ('C', selection_strategy_c),
]:
    idx = SAMPLE_INDICES[key]
    row = selected_rows.loc[idx]
    summary_rows.append({
        'Mẫu': key,
        'sample_id': SAMPLE_IDS[key],
        'idx': idx,
        'case_type': CASE_TYPES[key],
        'mae': float(row['mae']),
        'n_images': int(row['n_images']),
        'text_len': int(row['text_len']),
        'chiến_lược': strategy,
    })

selection_summary = pd.DataFrame(summary_rows)
print('\nDANH SÁCH 3 MẪU DEMO')
display(selection_summary.style.format({'mae': '{:.3f}'}))
print(selection_summary.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

# SHAP needs a multi-sample reference distribution. Reuse a cached Phase 4
# baseline when possible; otherwise build one from valid scanned samples.
SHAP_BACKGROUND = prepare_shap_background(
    df_valid.sort_values('mae')['idx'].tolist(),
    max_samples=8,
)
if SHAP_BACKGROUND is None:
    print('[SHAP] Không có baseline hợp lệ; các section SHAP sẽ skip an toàn.')

---
# 🧪 PHẦN 2 — MẪU A: DỰ ĐOÁN CHÍNH XÁC

## 2.1 · Tải mẫu & Dự đoán

In [ ]:
# ── Tải mẫu A và dự đoán ────────────────────────────────────────────────
SID_A  = SAMPLE_IDS['A']
IDX_A  = SAMPLE_INDICES['A']
CASE_A = CASE_TYPES['A']

gradcam_dir_A   = os.path.join(XAI_DIR, 'gradcam', SID_A)
attention_dir_A = os.path.join(XAI_DIR, 'attention', SID_A)
crossattn_dir_A = os.path.join(XAI_DIR, 'cross_attention', SID_A)
shap_dir_A      = os.path.join(XAI_DIR, 'shap', SID_A)
lime_dir_A      = os.path.join(XAI_DIR, 'lime', SID_A)
demo_dir_A      = os.path.join(DEMO_OUT, SID_A)
agent_dir_A     = os.path.join(AGENT_DIR, SID_A)
for output_dir in [
    gradcam_dir_A, attention_dir_A, crossattn_dir_A,
    shap_dir_A, lime_dir_A, demo_dir_A, agent_dir_A,
]:
    os.makedirs(output_dir, exist_ok=True)

print(f'Đang tải {SID_A} (idx={IDX_A}) ...')
sample_A = load_single_sample(
    csv_path=CSV_TEST,
    idx=IDX_A,
    tokenizer=tokenizer,
    image_processor=image_processor,
    image_dir=IMAGE_DIR,
    device=device,
)
print(
    f'  Text ({len(sample_A["text"])} ký tự): '
    f'{sample_A["text"][:150]} ...'
)
print(f'  Số ảnh thực: {sample_A["num_real_images"]}')
show_review_images(sample_A, SID_A)

pred_result_A = get_prediction(model, sample_A)
display_prediction_table(pred_result_A, SID_A)
plot_prediction_bars(
    pred_result_A,
    SID_A,
    save_path=os.path.join(demo_dir_A, f'{SID_A}_prediction.png'),
)

## 2.2 · Grad-CAM — Vùng ảnh quan trọng

> **Giới hạn:** image encoder dùng chung cho cả 5 đầu ra nên heatmap giữa các target có thể rất giống nhau. Demo chỉ hiển thị **Overall Satisfaction**, không dùng Grad-CAM để tuyên bố khác biệt per-target.

In [ ]:
# ── Grad-CAM: Only Overall Satisfaction (target_idx=4) ───────────────────────
# Shared encoder → cosine sim >0.95 across all 5 targets → show only overall
import datetime as _dt

TARGET_IDX_GRADCAM = 4
gradcam_results_A = {}
for img_idx in range(min(sample_A['num_real_images'], 4)):
    cam = run_safe(
        compute_gradcam_for_image, step_name=f'GradCAM img{img_idx}',
        fallback=None,
        model=model, sample=sample_A, target_idx=TARGET_IDX_GRADCAM,
        image_idx=img_idx, target_layer=target_layer, device=device,
    )
    gradcam_results_A[img_idx] = cam

n_show = min(sample_A['num_real_images'], 2)
if n_show > 0:
    import matplotlib.cm as _cm
    from PIL import Image as _PILI
    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show), squeeze=False)
    for img_idx in range(n_show):
        pil_img = sample_A['loaded_images'][img_idx]
        img224  = np.array(pil_img.convert('RGB').resize((224, 224)))
        cam     = gradcam_results_A.get(img_idx)

        axes[img_idx][0].imshow(img224)
        axes[img_idx][0].set_title(f'Ảnh {img_idx+1} — Gốc', fontsize=9)
        axes[img_idx][0].axis('off')

        if cam is not None:
            heatmap = _cm.jet(cam)[:, :, :3]
            axes[img_idx][1].imshow(heatmap)
            axes[img_idx][1].set_title('Grad-CAM Heatmap', fontsize=9)
        else:
            axes[img_idx][1].text(0.5, 0.5, 'N/A', ha='center', va='center',
                                   transform=axes[img_idx][1].transAxes)
        axes[img_idx][1].axis('off')

        if cam is not None:
            overlay = run_safe(overlay_cam_on_image, step_name='overlay',
                               cam=cam, original_image=pil_img,
                               image_size=224, colormap_name='jet', alpha=0.5)
            if overlay is not None:
                axes[img_idx][2].imshow(overlay)
                axes[img_idx][2].set_title('Overlay (CAM + Ảnh)', fontsize=9)
                cam_save = f'{gradcam_dir_A}/gradcam_img{img_idx}_overall.png'
                _PILI.fromarray(overlay).save(cam_save)
            else:
                axes[img_idx][2].axis('off')
        else:
            axes[img_idx][2].axis('off')

    fig.suptitle(f'Grad-CAM — {SID_A} (Overall Satisfaction)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    gcpath = f'{demo_dir_A}/{SID_A}_gradcam_3panel.png'
    fig.savefig(gcpath, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[XAI] Đã lưu: {gcpath}')
else:
    print('[SKIP] Không có ảnh → bỏ qua Grad-CAM')

# metadata.json cho EvidenceLoader (chỉ ghi nếu có ít nhất 1 ảnh được tính)
if any(v is not None for v in gradcam_results_A.values()):
    gradcam_meta_A = {
        'sample_id': SID_A,
        'sample_idx': IDX_A,
        'num_images': sample_A['num_real_images'],
        'num_targets': 1,
        'target_names': ['overall'],
        'display_names': ['Overall Satisfaction'],
        'target_layer': type(target_layer).__name__,
        'device': str(device),
        'timestamp': _dt.datetime.now().isoformat(),
        'artifacts': {
            f'img{i}_overall': f'gradcam_img{i}_overall.png'
            for i, v in gradcam_results_A.items() if v is not None
        },
    }
    with open(f'{gradcam_dir_A}/metadata.json', 'w', encoding='utf-8') as f:
        json.dump(gradcam_meta_A, f, ensure_ascii=False, indent=2)


## 2.3 · PhoBERT Attention — Mức từ đã gộp

> Output hiển thị đã gộp BPE/subword thành từ đọc được. Attention mô tả luồng thông tin, không tự nó chứng minh quan hệ nhân quả.

In [ ]:
# ── PhoBERT Attention: CLS → merged word-level importance ────────────────────
attn_result_A = run_safe(
    extract_phobert_attention,
    step_name='extract_phobert_attention',
    fallback=None,
    model=model,
    input_ids=sample_A['input_ids'],
    attention_mask=sample_A['attention_mask'],
    tokenizer=tokenizer,
)

word_importances_A = []
if attn_result_A is not None:
    attentions_A = attn_result_A['attentions']
    tokens_A = attn_result_A['tokens']
    seq_len_A = attn_result_A['seq_len']
    print(
        f'  tokens={seq_len_A}, '
        f'attention shape={attentions_A.shape}'
    )

    agg_matrix_A = aggregate_attention(
        attentions_A, strategy='last_layer_mean'
    )
    cls_result_A = cls_token_importance(
        agg_matrix_A, tokens_A
    )
    word_importances_A = merge_subword_attention(
        cls_result_A['importances'],
        tokens_A,
        strategy='mean',
    )

    print(f'Top 10 từ đã gộp subword ({SID_A}):')
    for word, score in word_importances_A[:10]:
        print(f'  {word:<24s} {score:.4f}')

    word_labels = [word for word, _ in word_importances_A]
    word_values = [value for _, value in word_importances_A]
    bar_path = os.path.join(
        attention_dir_A, 'cls_importance_word_bar.png'
    )
    bar_fig = run_safe(
        plot_cls_importance_bar,
        step_name='plot_word_attention',
        fallback=None,
        tokens=word_labels,
        importances=word_values,
        title=f'PhoBERT Word-Level Attention — {SID_A}',
        save_path=None,
        top_k=15,
        dpi=DEFAULT_DPI,
    )
    if bar_fig is not None:
        bar_fig.savefig(
            bar_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(bar_fig)
        print(f'[XAI] Đã lưu: {bar_path}')

    # Raw tensors are retained for audit/reuse, but are not shown to lecturers.
    np.savez_compressed(
        os.path.join(attention_dir_A, 'raw_attention.npz'),
        attentions=attentions_A,
        tokens=np.asarray(tokens_A, dtype=object),
    )
    word_payload = {
        'word_importances': [
            {'word': word, 'importance': float(score)}
            for word, score in word_importances_A
        ]
    }
    with open(
        os.path.join(attention_dir_A, 'word_importance.json'),
        'w',
        encoding='utf-8',
    ) as file:
        json.dump(word_payload, file, ensure_ascii=False, indent=2)
    with open(
        os.path.join(attention_dir_A, 'topk_tokens.json'),
        'w',
        encoding='utf-8',
    ) as file:
        json.dump(
            [
                {'token': word, 'importance': float(score)}
                for word, score in word_importances_A[:15]
            ],
            file,
            ensure_ascii=False,
            indent=2,
        )
    print(
        '[Attention] Visible output uses merged words; '
        'raw BPE/subword fragments are stored only as machine-readable data.'
    )
else:
    print('[SKIP] Không trích xuất được PhoBERT attention.')

## 2.4 · Cross-Attention — Tương tác hai chiều

- **Token → Patch:** khi xử lý một từ, mô hình phân bổ chú ý lên các patch ảnh nào?
- **Patch → Token:** từ một patch ảnh, mô hình liên kết ngược tới các từ nào?

Hai hướng dùng phép chuẩn hóa khác nhau; không diễn giải chúng như hai ma trận chuyển vị.

In [ ]:
# ── Bidirectional Cross-Attention: Token → Patch and Patch → Token ────────────
cross_result_A = run_safe(
    extract_cross_attention,
    step_name='extract_cross_attention',
    fallback=None,
    model=model,
    sample=sample_A,
    tokenizer=tokenizer,
)

t2i_A = None
i2t_A = None
if cross_result_A is not None:
    t2i_A = cross_result_A['t2i_attn']
    i2t_A = cross_result_A['i2t_attn']
    raw_tokens_A = cross_result_A['tokens']
    word_merge_A = run_safe(
        merge_cross_attention_words,
        step_name='merge_cross_attention_words',
        fallback=([], None, None),
        tokens=raw_tokens_A,
        t2i_attn=t2i_A,
        i2t_attn=i2t_A,
    )
    ca_words_A, t2i_words_A, i2t_words_A = word_merge_A

    if t2i_words_A is not None and i2t_words_A is not None:
        num_words, num_patches = t2i_words_A.shape
        grid_h = int(np.sqrt(num_patches))
        grid_w = (
            grid_h if grid_h * grid_h == num_patches
            else num_patches
        )
        patch_labels = (
            [
                f'({row},{col})'
                for row in range(grid_h)
                for col in range(grid_h)
            ]
            if grid_h * grid_h == num_patches
            else [str(index) for index in range(num_patches)]
        )

        word_strength = t2i_words_A.max(axis=1)
        top_word_indices = np.argsort(word_strength)[::-1][
            :min(20, num_words)
        ]
        patch_strength = i2t_words_A.max(axis=1)
        top_patch_indices = np.argsort(patch_strength)[::-1][
            :min(20, num_patches)
        ]

        visible_words = [ca_words_A[i] for i in top_word_indices]
        visible_patches = [patch_labels[i] for i in top_patch_indices]
        visible_t2i = t2i_words_A[top_word_indices, :]
        visible_i2t = i2t_words_A[
            np.ix_(top_patch_indices, top_word_indices)
        ]

        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        try:
            import seaborn as sns
            sns.heatmap(
                visible_t2i,
                xticklabels=patch_labels,
                yticklabels=visible_words,
                cmap='viridis',
                ax=axes[0],
                cbar_kws={'shrink': 0.65},
            )
            sns.heatmap(
                visible_i2t,
                xticklabels=visible_words,
                yticklabels=visible_patches,
                cmap='magma',
                ax=axes[1],
                cbar_kws={'shrink': 0.65},
            )
        except ImportError:
            axes[0].imshow(visible_t2i, aspect='auto', cmap='viridis')
            axes[1].imshow(visible_i2t, aspect='auto', cmap='magma')

        axes[0].set_title(
            'Token → Patch\nKhi đọc từ này, mô hình nhìn vùng ảnh nào?',
            fontsize=10,
            fontweight='bold',
        )
        axes[0].set_xlabel('Image patches')
        axes[0].set_ylabel('Merged words')
        axes[0].tick_params(axis='both', labelsize=6)

        axes[1].set_title(
            'Patch → Token\nKhi nhìn patch này, mô hình liên kết từ nào?',
            fontsize=10,
            fontweight='bold',
        )
        axes[1].set_xlabel('Merged words')
        axes[1].set_ylabel('Top image patches')
        axes[1].tick_params(axis='both', labelsize=7)

        fig.suptitle(
            f'Bidirectional Cross-Attention — {SID_A}',
            fontsize=13,
            fontweight='bold',
        )
        fig.tight_layout()
        cross_path = os.path.join(
            demo_dir_A, f'{SID_A}_cross_attention.png'
        )
        fig.savefig(
            cross_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(fig)
        print(f'[XAI] Đã lưu: {cross_path}')
        print(
            '[Giải thích hai chiều] Token→Patch cho biết vùng ảnh được truy vấn '
            'khi xử lý một từ; Patch→Token cho biết từ nào được một vùng ảnh '
            'liên kết. Hai ma trận dùng softmax theo hai trục khác nhau nên '
            'không được diễn giải như phép chuyển vị của nhau.'
        )

        np.savez_compressed(
            os.path.join(
                crossattn_dir_A, 'cross_attention_raw.npz'
            ),
            t2i=t2i_A,
            i2t=i2t_A,
            word_t2i=t2i_words_A,
            word_i2t=i2t_words_A,
            words=np.asarray(ca_words_A, dtype=object),
        )

        top5_indices = np.argsort(word_strength)[::-1][:5]
        entropy = -np.sum(
            t2i_words_A * np.log2(t2i_words_A + 1e-12),
            axis=1,
        )
        summary = {
            'sample_id': SID_A,
            'mean_token_entropy': float(entropy.mean()),
            'top_5_tokens': [
                {
                    'token': ca_words_A[i],
                    'importance': float(word_strength[i]),
                }
                for i in top5_indices
            ],
            'interpretation': {
                'token_to_patch': (
                    'For each merged word, distribution over image patches.'
                ),
                'patch_to_token': (
                    'For each image patch, distribution over merged words.'
                ),
            },
        }
        with open(
            os.path.join(
                crossattn_dir_A, 'cross_attention_summary.json'
            ),
            'w',
            encoding='utf-8',
        ) as file:
            json.dump(summary, file, ensure_ascii=False, indent=2)

        top_pairs = []
        for word_idx in top5_indices:
            patch_idx = int(np.argmax(t2i_words_A[word_idx]))
            top_pairs.append({
                'token': ca_words_A[word_idx],
                'patch_row': (
                    patch_idx // grid_h if grid_h * grid_h == num_patches
                    else 0
                ),
                'patch_col': (
                    patch_idx % grid_h if grid_h * grid_h == num_patches
                    else patch_idx
                ),
                'attention': float(
                    t2i_words_A[word_idx, patch_idx]
                ),
            })
        with open(
            os.path.join(crossattn_dir_A, 'token_patch_topk.json'),
            'w',
            encoding='utf-8',
        ) as file:
            json.dump(top_pairs, file, ensure_ascii=False, indent=2)
    else:
        print('[SKIP] Không thể gộp Cross-Attention ở mức từ.')
else:
    print('[SKIP] Không trích xuất được Cross-Attention.')

## 2.5 · SHAP — Đóng góp fused embedding [1024]

> `text-origin` (dims 0:512) và `image-origin` (dims 512:1024) đều là biểu diễn **sau Cross-Attention**. Đây không phải hai modality thuần túy. SHAP dùng baseline nhiều mẫu, không dùng chính sample làm baseline.

In [ ]:
# ── SHAP on fused embedding [1024], using a multi-sample baseline ─────────────
fused_result_A = run_safe(
    extract_fused_embeddings,
    step_name='extract_fused_embeddings',
    fallback=(None, None, None),
    model=model,
    dataloader=[sample_to_batch(sample_A)],
    device=device,
    max_samples=1,
)
fused_A = fused_result_A[0] if fused_result_A else None

shap_values_by_factor_A = {}
shap_contrib_by_factor_A = {}
shap_vals_A = None
shap_contrib_A = None

if fused_A is None:
    print('[SKIP] Không extract được fused embedding.')
elif SHAP_BACKGROUND is None:
    print(
        '[SKIP] SHAP không có multi-sample background hợp lệ; '
        'không dùng chính sample làm baseline vì sẽ tạo attribution bằng 0.'
    )
else:
    print(
        f'[SHAP] Background={tuple(SHAP_BACKGROUND.shape)}, '
        f'sample={tuple(fused_A.shape)}'
    )
    for score_index, factor_name in enumerate(FACTOR_NAMES):
        wrapper = FusionHeadWrapper(model.head, score_index=score_index)
        shap_result = run_safe(
            compute_shap_values,
            step_name=f'SHAP_{factor_name}',
            fallback=(None, None),
            wrapper=wrapper,
            background=SHAP_BACKGROUND,
            samples=fused_A,
        )
        values = shap_result[0] if shap_result else None
        if values is None or len(values) == 0:
            continue
        sample_values = np.asarray(values[0], dtype=float).reshape(-1)
        if sample_values.size != FUSED_DIM or not np.isfinite(
            sample_values
        ).all():
            print(
                f'[SKIP] SHAP {factor_name}: invalid shape/value '
                f'{sample_values.shape}'
            )
            continue
        contribution = modality_contribution(sample_values)
        shap_values_by_factor_A[factor_name] = sample_values
        shap_contrib_by_factor_A[factor_name] = contribution
        print(
            f'  {DISPLAY_NAMES[score_index]:<22s} '
            f'text-origin={contribution["text_pct"]:5.1f}% | '
            f'image-origin={contribution["image_pct"]:5.1f}%'
        )

    shap_vals_A = shap_values_by_factor_A.get('overall')
    shap_contrib_A = shap_contrib_by_factor_A.get('overall')

    if shap_contrib_by_factor_A:
        contribution_path = os.path.join(
            shap_dir_A, 'shap_modality_contribution.json'
        )
        with open(
            contribution_path, 'w', encoding='utf-8'
        ) as file:
            json.dump(
                shap_contrib_by_factor_A,
                file,
                ensure_ascii=False,
                indent=2,
            )
        np.savez_compressed(
            os.path.join(shap_dir_A, 'raw_shap_values.npz'),
            **shap_values_by_factor_A,
        )
        print(f'[XAI] Đã lưu: {contribution_path}')

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        if (
            shap_contrib_A is not None
            and shap_contrib_A['text_abs']
            + shap_contrib_A['image_abs'] > 1e-12
        ):
            axes[0].pie(
                [
                    shap_contrib_A['text_abs'],
                    shap_contrib_A['image_abs'],
                ],
                labels=[
                    f'Text-origin\n{shap_contrib_A["text_pct"]:.1f}%',
                    f'Image-origin\n{shap_contrib_A["image_pct"]:.1f}%',
                ],
                colors=[
                    COLOR_SCHEMES['modality_colors']['text'],
                    COLOR_SCHEMES['modality_colors']['image'],
                ],
                autopct='%1.1f%%',
                startangle=90,
            )
            axes[0].set_title(
                'Overall Satisfaction\nModality contribution',
                fontweight='bold',
            )
        else:
            axes[0].text(
                0.5, 0.5, 'Overall SHAP unavailable',
                ha='center', va='center', transform=axes[0].transAxes
            )
            axes[0].axis('off')

        available_factors = [
            factor for factor in FACTOR_NAMES
            if factor in shap_contrib_by_factor_A
        ]
        x_positions = np.arange(len(available_factors))
        text_pct = [
            shap_contrib_by_factor_A[factor]['text_pct']
            for factor in available_factors
        ]
        image_pct = [
            shap_contrib_by_factor_A[factor]['image_pct']
            for factor in available_factors
        ]
        axes[1].bar(
            x_positions,
            text_pct,
            label='Text-origin',
            color=COLOR_SCHEMES['modality_colors']['text'],
        )
        axes[1].bar(
            x_positions,
            image_pct,
            bottom=text_pct,
            label='Image-origin',
            color=COLOR_SCHEMES['modality_colors']['image'],
        )
        axes[1].set_xticks(x_positions)
        axes[1].set_xticklabels(
            [
                DISPLAY_NAMES[FACTOR_NAMES.index(factor)]
                for factor in available_factors
            ],
            rotation=25,
            ha='right',
            fontsize=8,
        )
        axes[1].set_ylim(0, 100)
        axes[1].set_ylabel('|SHAP| contribution (%)')
        axes[1].set_title(
            'Per-target origin contribution', fontweight='bold'
        )
        axes[1].legend(fontsize=8)

        if shap_vals_A is not None:
            top_count = min(20, shap_vals_A.size)
            top_indices = np.argsort(np.abs(shap_vals_A))[
                -top_count:
            ][::-1]
            top_values = shap_vals_A[top_indices]
            dim_labels = [
                (
                    f'T{index}'
                    if index < CROSS_ATTN_HIDDEN_DIM
                    else f'I{index - CROSS_ATTN_HIDDEN_DIM}'
                )
                for index in top_indices
            ]
            colors = [
                (
                    COLOR_SCHEMES['shap_positive']
                    if value >= 0
                    else COLOR_SCHEMES['shap_negative']
                )
                for value in top_values
            ]
            axes[2].barh(
                range(top_count),
                top_values[::-1],
                color=colors[::-1],
            )
            axes[2].set_yticks(range(top_count))
            axes[2].set_yticklabels(dim_labels[::-1], fontsize=7)
            axes[2].axvline(0, color='black', linewidth=0.8)
            axes[2].set_xlabel('SHAP value')
            axes[2].set_title(
                'Overall: top fused dimensions', fontweight='bold'
            )
        else:
            axes[2].text(
                0.5, 0.5, 'Overall SHAP unavailable',
                ha='center', va='center', transform=axes[2].transAxes
            )
            axes[2].axis('off')

        fig.suptitle(
            f'SHAP — {SID_A}\n'
            'Origins are cross-attended representations, not pure modalities',
            fontsize=12,
            fontweight='bold',
        )
        fig.tight_layout()
        shap_path = os.path.join(
            demo_dir_A, f'{SID_A}_shap_analysis.png'
        )
        fig.savefig(
            shap_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(fig)
        print(f'[XAI] Đã lưu: {shap_path}')
    else:
        print('[SKIP] Không target nào tạo được SHAP values hợp lệ.')

## 2.6 · LIME — Giải thích cục bộ (Text + Image)

> LIME đo độ nhạy quanh **mẫu hiện tại** bằng perturbation ngẫu nhiên. Kết quả không đại diện cho hành vi toàn cục của mô hình.

In [ ]:
# ── LIME: Text + Image (target=Overall Satisfaction) ─────────────────────────
TARGET_IDX_LIME = 4
lime_text_weights_A = {}
lime_text_exp_A     = None
lime_image_exp_A    = None
lime_img_paths_A    = {}

# LIME Text
print('[LIME Text] Đang tính ...')
lime_text_exp_A = run_safe(
    run_lime_text, step_name='LIME_text',
    fallback=None,
    model=model, sample=sample_A, score_index=TARGET_IDX_LIME,
    tokenizer=tokenizer, device=device,
    num_features=10, num_samples=300,
)
raw_weights_A = []
if lime_text_exp_A is not None:
    raw_weights_A = lime_text_exp_A.as_list(label=1)
    lime_text_weights_A = dict(raw_weights_A)
    print('  Top LIME words:')
    for word, w in sorted(raw_weights_A, key=lambda x: abs(x[1]), reverse=True)[:8]:
        print(f'    {("+" if w>0 else "-")} {word:<18s} {abs(w):.4f}')
    lime_factor_name = FACTOR_NAMES[TARGET_IDX_LIME]
    lt_path = f'{lime_dir_A}/{SID_A}_lime_text_{lime_factor_name}_weights.json'
    with open(lt_path, 'w', encoding='utf-8') as f:
        json.dump(raw_weights_A, f, ensure_ascii=False, indent=2)
    print(f'[XAI] Đã lưu: {lt_path}')
    lime_text_bar_A = os.path.join(
        lime_dir_A,
        f'{SID_A}_lime_text_{lime_factor_name}_bar.png',
    )
    save_lime_text_bar(
        raw_weights_A,
        lime_text_bar_A,
        f'LIME Local Text Explanation — {SID_A}',
    )
    print(f'[XAI] Đã lưu: {lime_text_bar_A}')
else:
    print('[SKIP] LIME text thất bại.')

# LIME Image
if sample_A['num_real_images'] > 0:
    print('[LIME Image] Đang tính ...')
    lime_image_exp_A = run_safe(
        run_lime_image, step_name='LIME_image',
        fallback=None,
        model=model, sample=sample_A, score_index=TARGET_IDX_LIME,
        image_processor=image_processor, device=device, num_samples=300,
    )
    if lime_image_exp_A is not None:
        lime_img_paths_A = run_safe(
            save_lime_image_explanation, step_name='save_LIME_image',
            fallback={},
            explanation=lime_image_exp_A,
            original_image=sample_A['loaded_images'][0],
            save_dir=lime_dir_A,
            sample_id=SID_A,
            target_idx=TARGET_IDX_LIME,
            factor_name=FACTOR_NAMES[TARGET_IDX_LIME],
            dpi=DEFAULT_DPI,
        ) or {}
    else:
        print('[SKIP] LIME image thất bại.')
else:
    print('[SKIP] Không có ảnh → bỏ qua LIME image.')

# 4-panel LIME visualisation
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1: text importance bar
if lime_text_weights_A:
    w_sorted = sorted(lime_text_weights_A.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
    w_names  = [w[0] for w in reversed(w_sorted)]
    w_values = [w[1] for w in reversed(w_sorted)]
    bar_cols = [COLOR_SCHEMES['shap_positive'] if v >= 0
                else COLOR_SCHEMES['shap_negative'] for v in w_values]
    axes[0].barh(range(len(w_names)), w_values, color=bar_cols)
    axes[0].set_yticks(range(len(w_names)))
    axes[0].set_yticklabels(w_names, fontsize=8)
    axes[0].axvline(0, color='black', lw=0.8)
    axes[0].set_title('LIME Text\n(từ quan trọng)', fontsize=9, fontweight='bold')
    axes[0].set_xlabel('LIME weight', fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'N/A', ha='center', va='center',
                 transform=axes[0].transAxes); axes[0].axis('off')

# Panels 2-4: image overlays
from PIL import Image as _PILImg3
for col_i, (key, title) in enumerate(
        [('positive_overlay','LIME Image (+)'),
         ('negative_overlay','LIME Image (−)'),
         ('combined_overlay','LIME Image (±)')]):
    ax = axes[col_i + 1]
    path = lime_img_paths_A.get(key)
    if path and os.path.exists(path):
        ax.imshow(np.array(_PILImg3.open(path)))
        ax.axis('off')
        ax.set_title(title, fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes); ax.axis('off')

fig.suptitle(f'LIME Explanation — {SID_A} (Overall Satisfaction)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
lime_4p = f'{demo_dir_A}/{SID_A}_lime_4panel.png'
fig.savefig(lime_4p, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[XAI] Đã lưu: {lime_4p}')


## 2.7 · AI Agent — Báo cáo tổng hợp

> GPT-4o chỉ verbalize evidence đã tạo và chỉ được gọi khi `OPENAI_API_KEY` tồn tại trong environment/Colab Secrets. Thiếu key thì section được bỏ qua an toàn.

In [ ]:
# ── Kiểm tra artifact & AI Agent ─────────────────────────────────────────────
artifact_check_A = check_sample_artifacts(SID_A, XAI_DIR)

add_cross_method_row(
    SID_A, pred_result_A,
    shap_contrib_A,
    raw_weights_A,
    case_type=CASE_A,
)

agent_output_A = None
if AGENT_AVAILABLE:
    agent_config = AgentConfig(language='vi')
    if agent_config.api_key:
        print(f'[Agent] Đang chạy AI Agent cho {SID_A} ...')
        agent_A = ExplanationAgent(agent_config)
        agent_output_A = run_safe(
            agent_A.explain_sample, step_name='AI_Agent',
            fallback=None,
            sample_id=SID_A,
            review_text=sample_A['text'],
            predictions=pred_result_A['predictions'],
            xai_dir=XAI_DIR,
            ground_truth=pred_result_A['ground_truth'],
            case_type=CASE_A,
            language='vi',
            mode='text_only',
            num_images=sample_A['num_real_images'],
            output_dir=agent_dir_A,
        )
        if agent_output_A:
            print(f"[Agent] ✅ Evidence completeness: {agent_output_A.get('evidence_completeness','N/A')}")
        else:
            print('[Agent] Không tạo được báo cáo.')
    else:
        print('[Agent] OPENAI_API_KEY không tìm thấy → bỏ qua.')
        print('  → Add OPENAI_API_KEY in Colab Secrets, then rerun this cell.')
else:
    print('[Agent] Module không khả dụng.')

print_sample_summary(SID_A, pred_result_A, artifact_check_A)
print(f'✅ HOÀN THÀNH MẪU A: {SID_A}')

---
# 🧪 PHẦN 3 — MẪU B: LỖI / XUNG ĐỘT

## 3.1 · Tải mẫu & Dự đoán

In [ ]:
# ── Tải mẫu B và dự đoán ────────────────────────────────────────────────
SID_B  = SAMPLE_IDS['B']
IDX_B  = SAMPLE_INDICES['B']
CASE_B = CASE_TYPES['B']

gradcam_dir_B   = os.path.join(XAI_DIR, 'gradcam', SID_B)
attention_dir_B = os.path.join(XAI_DIR, 'attention', SID_B)
crossattn_dir_B = os.path.join(XAI_DIR, 'cross_attention', SID_B)
shap_dir_B      = os.path.join(XAI_DIR, 'shap', SID_B)
lime_dir_B      = os.path.join(XAI_DIR, 'lime', SID_B)
demo_dir_B      = os.path.join(DEMO_OUT, SID_B)
agent_dir_B     = os.path.join(AGENT_DIR, SID_B)
for output_dir in [
    gradcam_dir_B, attention_dir_B, crossattn_dir_B,
    shap_dir_B, lime_dir_B, demo_dir_B, agent_dir_B,
]:
    os.makedirs(output_dir, exist_ok=True)

print(f'Đang tải {SID_B} (idx={IDX_B}) ...')
sample_B = load_single_sample(
    csv_path=CSV_TEST,
    idx=IDX_B,
    tokenizer=tokenizer,
    image_processor=image_processor,
    image_dir=IMAGE_DIR,
    device=device,
)
print(
    f'  Text ({len(sample_B["text"])} ký tự): '
    f'{sample_B["text"][:150]} ...'
)
print(f'  Số ảnh thực: {sample_B["num_real_images"]}')
show_review_images(sample_B, SID_B)

pred_result_B = get_prediction(model, sample_B)
display_prediction_table(pred_result_B, SID_B)
plot_prediction_bars(
    pred_result_B,
    SID_B,
    save_path=os.path.join(demo_dir_B, f'{SID_B}_prediction.png'),
)

## 3.2 · Grad-CAM — Vùng ảnh quan trọng

> **Giới hạn:** image encoder dùng chung cho cả 5 đầu ra nên heatmap giữa các target có thể rất giống nhau. Demo chỉ hiển thị **Overall Satisfaction**, không dùng Grad-CAM để tuyên bố khác biệt per-target.

In [ ]:
# ── Grad-CAM: Only Overall Satisfaction (target_idx=4) ───────────────────────
# Shared encoder → cosine sim >0.95 across all 5 targets → show only overall
import datetime as _dt

TARGET_IDX_GRADCAM = 4
gradcam_results_B = {}
for img_idx in range(min(sample_B['num_real_images'], 4)):
    cam = run_safe(
        compute_gradcam_for_image, step_name=f'GradCAM img{img_idx}',
        fallback=None,
        model=model, sample=sample_B, target_idx=TARGET_IDX_GRADCAM,
        image_idx=img_idx, target_layer=target_layer, device=device,
    )
    gradcam_results_B[img_idx] = cam

n_show = min(sample_B['num_real_images'], 2)
if n_show > 0:
    import matplotlib.cm as _cm
    from PIL import Image as _PILI
    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show), squeeze=False)
    for img_idx in range(n_show):
        pil_img = sample_B['loaded_images'][img_idx]
        img224  = np.array(pil_img.convert('RGB').resize((224, 224)))
        cam     = gradcam_results_B.get(img_idx)

        axes[img_idx][0].imshow(img224)
        axes[img_idx][0].set_title(f'Ảnh {img_idx+1} — Gốc', fontsize=9)
        axes[img_idx][0].axis('off')

        if cam is not None:
            heatmap = _cm.jet(cam)[:, :, :3]
            axes[img_idx][1].imshow(heatmap)
            axes[img_idx][1].set_title('Grad-CAM Heatmap', fontsize=9)
        else:
            axes[img_idx][1].text(0.5, 0.5, 'N/A', ha='center', va='center',
                                   transform=axes[img_idx][1].transAxes)
        axes[img_idx][1].axis('off')

        if cam is not None:
            overlay = run_safe(overlay_cam_on_image, step_name='overlay',
                               cam=cam, original_image=pil_img,
                               image_size=224, colormap_name='jet', alpha=0.5)
            if overlay is not None:
                axes[img_idx][2].imshow(overlay)
                axes[img_idx][2].set_title('Overlay (CAM + Ảnh)', fontsize=9)
                cam_save = f'{gradcam_dir_B}/gradcam_img{img_idx}_overall.png'
                _PILI.fromarray(overlay).save(cam_save)
            else:
                axes[img_idx][2].axis('off')
        else:
            axes[img_idx][2].axis('off')

    fig.suptitle(f'Grad-CAM — {SID_B} (Overall Satisfaction)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    gcpath = f'{demo_dir_B}/{SID_B}_gradcam_3panel.png'
    fig.savefig(gcpath, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[XAI] Đã lưu: {gcpath}')
else:
    print('[SKIP] Không có ảnh → bỏ qua Grad-CAM')

# metadata.json cho EvidenceLoader (chỉ ghi nếu có ít nhất 1 ảnh được tính)
if any(v is not None for v in gradcam_results_B.values()):
    gradcam_meta_B = {
        'sample_id': SID_B,
        'sample_idx': IDX_B,
        'num_images': sample_B['num_real_images'],
        'num_targets': 1,
        'target_names': ['overall'],
        'display_names': ['Overall Satisfaction'],
        'target_layer': type(target_layer).__name__,
        'device': str(device),
        'timestamp': _dt.datetime.now().isoformat(),
        'artifacts': {
            f'img{i}_overall': f'gradcam_img{i}_overall.png'
            for i, v in gradcam_results_B.items() if v is not None
        },
    }
    with open(f'{gradcam_dir_B}/metadata.json', 'w', encoding='utf-8') as f:
        json.dump(gradcam_meta_B, f, ensure_ascii=False, indent=2)


## 3.3 · PhoBERT Attention — Mức từ đã gộp

> Output hiển thị đã gộp BPE/subword thành từ đọc được. Attention mô tả luồng thông tin, không tự nó chứng minh quan hệ nhân quả.

In [ ]:
# ── PhoBERT Attention: CLS → merged word-level importance ────────────────────
attn_result_B = run_safe(
    extract_phobert_attention,
    step_name='extract_phobert_attention',
    fallback=None,
    model=model,
    input_ids=sample_B['input_ids'],
    attention_mask=sample_B['attention_mask'],
    tokenizer=tokenizer,
)

word_importances_B = []
if attn_result_B is not None:
    attentions_B = attn_result_B['attentions']
    tokens_B = attn_result_B['tokens']
    seq_len_B = attn_result_B['seq_len']
    print(
        f'  tokens={seq_len_B}, '
        f'attention shape={attentions_B.shape}'
    )

    agg_matrix_B = aggregate_attention(
        attentions_B, strategy='last_layer_mean'
    )
    cls_result_B = cls_token_importance(
        agg_matrix_B, tokens_B
    )
    word_importances_B = merge_subword_attention(
        cls_result_B['importances'],
        tokens_B,
        strategy='mean',
    )

    print(f'Top 10 từ đã gộp subword ({SID_B}):')
    for word, score in word_importances_B[:10]:
        print(f'  {word:<24s} {score:.4f}')

    word_labels = [word for word, _ in word_importances_B]
    word_values = [value for _, value in word_importances_B]
    bar_path = os.path.join(
        attention_dir_B, 'cls_importance_word_bar.png'
    )
    bar_fig = run_safe(
        plot_cls_importance_bar,
        step_name='plot_word_attention',
        fallback=None,
        tokens=word_labels,
        importances=word_values,
        title=f'PhoBERT Word-Level Attention — {SID_B}',
        save_path=None,
        top_k=15,
        dpi=DEFAULT_DPI,
    )
    if bar_fig is not None:
        bar_fig.savefig(
            bar_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(bar_fig)
        print(f'[XAI] Đã lưu: {bar_path}')

    # Raw tensors are retained for audit/reuse, but are not shown to lecturers.
    np.savez_compressed(
        os.path.join(attention_dir_B, 'raw_attention.npz'),
        attentions=attentions_B,
        tokens=np.asarray(tokens_B, dtype=object),
    )
    word_payload = {
        'word_importances': [
            {'word': word, 'importance': float(score)}
            for word, score in word_importances_B
        ]
    }
    with open(
        os.path.join(attention_dir_B, 'word_importance.json'),
        'w',
        encoding='utf-8',
    ) as file:
        json.dump(word_payload, file, ensure_ascii=False, indent=2)
    with open(
        os.path.join(attention_dir_B, 'topk_tokens.json'),
        'w',
        encoding='utf-8',
    ) as file:
        json.dump(
            [
                {'token': word, 'importance': float(score)}
                for word, score in word_importances_B[:15]
            ],
            file,
            ensure_ascii=False,
            indent=2,
        )
    print(
        '[Attention] Visible output uses merged words; '
        'raw BPE/subword fragments are stored only as machine-readable data.'
    )
else:
    print('[SKIP] Không trích xuất được PhoBERT attention.')

## 3.4 · Cross-Attention — Tương tác hai chiều

- **Token → Patch:** khi xử lý một từ, mô hình phân bổ chú ý lên các patch ảnh nào?
- **Patch → Token:** từ một patch ảnh, mô hình liên kết ngược tới các từ nào?

Hai hướng dùng phép chuẩn hóa khác nhau; không diễn giải chúng như hai ma trận chuyển vị.

In [ ]:
# ── Bidirectional Cross-Attention: Token → Patch and Patch → Token ────────────
cross_result_B = run_safe(
    extract_cross_attention,
    step_name='extract_cross_attention',
    fallback=None,
    model=model,
    sample=sample_B,
    tokenizer=tokenizer,
)

t2i_B = None
i2t_B = None
if cross_result_B is not None:
    t2i_B = cross_result_B['t2i_attn']
    i2t_B = cross_result_B['i2t_attn']
    raw_tokens_B = cross_result_B['tokens']
    word_merge_B = run_safe(
        merge_cross_attention_words,
        step_name='merge_cross_attention_words',
        fallback=([], None, None),
        tokens=raw_tokens_B,
        t2i_attn=t2i_B,
        i2t_attn=i2t_B,
    )
    ca_words_B, t2i_words_B, i2t_words_B = word_merge_B

    if t2i_words_B is not None and i2t_words_B is not None:
        num_words, num_patches = t2i_words_B.shape
        grid_h = int(np.sqrt(num_patches))
        grid_w = (
            grid_h if grid_h * grid_h == num_patches
            else num_patches
        )
        patch_labels = (
            [
                f'({row},{col})'
                for row in range(grid_h)
                for col in range(grid_h)
            ]
            if grid_h * grid_h == num_patches
            else [str(index) for index in range(num_patches)]
        )

        word_strength = t2i_words_B.max(axis=1)
        top_word_indices = np.argsort(word_strength)[::-1][
            :min(20, num_words)
        ]
        patch_strength = i2t_words_B.max(axis=1)
        top_patch_indices = np.argsort(patch_strength)[::-1][
            :min(20, num_patches)
        ]

        visible_words = [ca_words_B[i] for i in top_word_indices]
        visible_patches = [patch_labels[i] for i in top_patch_indices]
        visible_t2i = t2i_words_B[top_word_indices, :]
        visible_i2t = i2t_words_B[
            np.ix_(top_patch_indices, top_word_indices)
        ]

        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        try:
            import seaborn as sns
            sns.heatmap(
                visible_t2i,
                xticklabels=patch_labels,
                yticklabels=visible_words,
                cmap='viridis',
                ax=axes[0],
                cbar_kws={'shrink': 0.65},
            )
            sns.heatmap(
                visible_i2t,
                xticklabels=visible_words,
                yticklabels=visible_patches,
                cmap='magma',
                ax=axes[1],
                cbar_kws={'shrink': 0.65},
            )
        except ImportError:
            axes[0].imshow(visible_t2i, aspect='auto', cmap='viridis')
            axes[1].imshow(visible_i2t, aspect='auto', cmap='magma')

        axes[0].set_title(
            'Token → Patch\nKhi đọc từ này, mô hình nhìn vùng ảnh nào?',
            fontsize=10,
            fontweight='bold',
        )
        axes[0].set_xlabel('Image patches')
        axes[0].set_ylabel('Merged words')
        axes[0].tick_params(axis='both', labelsize=6)

        axes[1].set_title(
            'Patch → Token\nKhi nhìn patch này, mô hình liên kết từ nào?',
            fontsize=10,
            fontweight='bold',
        )
        axes[1].set_xlabel('Merged words')
        axes[1].set_ylabel('Top image patches')
        axes[1].tick_params(axis='both', labelsize=7)

        fig.suptitle(
            f'Bidirectional Cross-Attention — {SID_B}',
            fontsize=13,
            fontweight='bold',
        )
        fig.tight_layout()
        cross_path = os.path.join(
            demo_dir_B, f'{SID_B}_cross_attention.png'
        )
        fig.savefig(
            cross_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(fig)
        print(f'[XAI] Đã lưu: {cross_path}')
        print(
            '[Giải thích hai chiều] Token→Patch cho biết vùng ảnh được truy vấn '
            'khi xử lý một từ; Patch→Token cho biết từ nào được một vùng ảnh '
            'liên kết. Hai ma trận dùng softmax theo hai trục khác nhau nên '
            'không được diễn giải như phép chuyển vị của nhau.'
        )

        np.savez_compressed(
            os.path.join(
                crossattn_dir_B, 'cross_attention_raw.npz'
            ),
            t2i=t2i_B,
            i2t=i2t_B,
            word_t2i=t2i_words_B,
            word_i2t=i2t_words_B,
            words=np.asarray(ca_words_B, dtype=object),
        )

        top5_indices = np.argsort(word_strength)[::-1][:5]
        entropy = -np.sum(
            t2i_words_B * np.log2(t2i_words_B + 1e-12),
            axis=1,
        )
        summary = {
            'sample_id': SID_B,
            'mean_token_entropy': float(entropy.mean()),
            'top_5_tokens': [
                {
                    'token': ca_words_B[i],
                    'importance': float(word_strength[i]),
                }
                for i in top5_indices
            ],
            'interpretation': {
                'token_to_patch': (
                    'For each merged word, distribution over image patches.'
                ),
                'patch_to_token': (
                    'For each image patch, distribution over merged words.'
                ),
            },
        }
        with open(
            os.path.join(
                crossattn_dir_B, 'cross_attention_summary.json'
            ),
            'w',
            encoding='utf-8',
        ) as file:
            json.dump(summary, file, ensure_ascii=False, indent=2)

        top_pairs = []
        for word_idx in top5_indices:
            patch_idx = int(np.argmax(t2i_words_B[word_idx]))
            top_pairs.append({
                'token': ca_words_B[word_idx],
                'patch_row': (
                    patch_idx // grid_h if grid_h * grid_h == num_patches
                    else 0
                ),
                'patch_col': (
                    patch_idx % grid_h if grid_h * grid_h == num_patches
                    else patch_idx
                ),
                'attention': float(
                    t2i_words_B[word_idx, patch_idx]
                ),
            })
        with open(
            os.path.join(crossattn_dir_B, 'token_patch_topk.json'),
            'w',
            encoding='utf-8',
        ) as file:
            json.dump(top_pairs, file, ensure_ascii=False, indent=2)
    else:
        print('[SKIP] Không thể gộp Cross-Attention ở mức từ.')
else:
    print('[SKIP] Không trích xuất được Cross-Attention.')

## 3.5 · SHAP — Đóng góp fused embedding [1024]

> `text-origin` (dims 0:512) và `image-origin` (dims 512:1024) đều là biểu diễn **sau Cross-Attention**. Đây không phải hai modality thuần túy. SHAP dùng baseline nhiều mẫu, không dùng chính sample làm baseline.

In [ ]:
# ── SHAP on fused embedding [1024], using a multi-sample baseline ─────────────
fused_result_B = run_safe(
    extract_fused_embeddings,
    step_name='extract_fused_embeddings',
    fallback=(None, None, None),
    model=model,
    dataloader=[sample_to_batch(sample_B)],
    device=device,
    max_samples=1,
)
fused_B = fused_result_B[0] if fused_result_B else None

shap_values_by_factor_B = {}
shap_contrib_by_factor_B = {}
shap_vals_B = None
shap_contrib_B = None

if fused_B is None:
    print('[SKIP] Không extract được fused embedding.')
elif SHAP_BACKGROUND is None:
    print(
        '[SKIP] SHAP không có multi-sample background hợp lệ; '
        'không dùng chính sample làm baseline vì sẽ tạo attribution bằng 0.'
    )
else:
    print(
        f'[SHAP] Background={tuple(SHAP_BACKGROUND.shape)}, '
        f'sample={tuple(fused_B.shape)}'
    )
    for score_index, factor_name in enumerate(FACTOR_NAMES):
        wrapper = FusionHeadWrapper(model.head, score_index=score_index)
        shap_result = run_safe(
            compute_shap_values,
            step_name=f'SHAP_{factor_name}',
            fallback=(None, None),
            wrapper=wrapper,
            background=SHAP_BACKGROUND,
            samples=fused_B,
        )
        values = shap_result[0] if shap_result else None
        if values is None or len(values) == 0:
            continue
        sample_values = np.asarray(values[0], dtype=float).reshape(-1)
        if sample_values.size != FUSED_DIM or not np.isfinite(
            sample_values
        ).all():
            print(
                f'[SKIP] SHAP {factor_name}: invalid shape/value '
                f'{sample_values.shape}'
            )
            continue
        contribution = modality_contribution(sample_values)
        shap_values_by_factor_B[factor_name] = sample_values
        shap_contrib_by_factor_B[factor_name] = contribution
        print(
            f'  {DISPLAY_NAMES[score_index]:<22s} '
            f'text-origin={contribution["text_pct"]:5.1f}% | '
            f'image-origin={contribution["image_pct"]:5.1f}%'
        )

    shap_vals_B = shap_values_by_factor_B.get('overall')
    shap_contrib_B = shap_contrib_by_factor_B.get('overall')

    if shap_contrib_by_factor_B:
        contribution_path = os.path.join(
            shap_dir_B, 'shap_modality_contribution.json'
        )
        with open(
            contribution_path, 'w', encoding='utf-8'
        ) as file:
            json.dump(
                shap_contrib_by_factor_B,
                file,
                ensure_ascii=False,
                indent=2,
            )
        np.savez_compressed(
            os.path.join(shap_dir_B, 'raw_shap_values.npz'),
            **shap_values_by_factor_B,
        )
        print(f'[XAI] Đã lưu: {contribution_path}')

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        if (
            shap_contrib_B is not None
            and shap_contrib_B['text_abs']
            + shap_contrib_B['image_abs'] > 1e-12
        ):
            axes[0].pie(
                [
                    shap_contrib_B['text_abs'],
                    shap_contrib_B['image_abs'],
                ],
                labels=[
                    f'Text-origin\n{shap_contrib_B["text_pct"]:.1f}%',
                    f'Image-origin\n{shap_contrib_B["image_pct"]:.1f}%',
                ],
                colors=[
                    COLOR_SCHEMES['modality_colors']['text'],
                    COLOR_SCHEMES['modality_colors']['image'],
                ],
                autopct='%1.1f%%',
                startangle=90,
            )
            axes[0].set_title(
                'Overall Satisfaction\nModality contribution',
                fontweight='bold',
            )
        else:
            axes[0].text(
                0.5, 0.5, 'Overall SHAP unavailable',
                ha='center', va='center', transform=axes[0].transAxes
            )
            axes[0].axis('off')

        available_factors = [
            factor for factor in FACTOR_NAMES
            if factor in shap_contrib_by_factor_B
        ]
        x_positions = np.arange(len(available_factors))
        text_pct = [
            shap_contrib_by_factor_B[factor]['text_pct']
            for factor in available_factors
        ]
        image_pct = [
            shap_contrib_by_factor_B[factor]['image_pct']
            for factor in available_factors
        ]
        axes[1].bar(
            x_positions,
            text_pct,
            label='Text-origin',
            color=COLOR_SCHEMES['modality_colors']['text'],
        )
        axes[1].bar(
            x_positions,
            image_pct,
            bottom=text_pct,
            label='Image-origin',
            color=COLOR_SCHEMES['modality_colors']['image'],
        )
        axes[1].set_xticks(x_positions)
        axes[1].set_xticklabels(
            [
                DISPLAY_NAMES[FACTOR_NAMES.index(factor)]
                for factor in available_factors
            ],
            rotation=25,
            ha='right',
            fontsize=8,
        )
        axes[1].set_ylim(0, 100)
        axes[1].set_ylabel('|SHAP| contribution (%)')
        axes[1].set_title(
            'Per-target origin contribution', fontweight='bold'
        )
        axes[1].legend(fontsize=8)

        if shap_vals_B is not None:
            top_count = min(20, shap_vals_B.size)
            top_indices = np.argsort(np.abs(shap_vals_B))[
                -top_count:
            ][::-1]
            top_values = shap_vals_B[top_indices]
            dim_labels = [
                (
                    f'T{index}'
                    if index < CROSS_ATTN_HIDDEN_DIM
                    else f'I{index - CROSS_ATTN_HIDDEN_DIM}'
                )
                for index in top_indices
            ]
            colors = [
                (
                    COLOR_SCHEMES['shap_positive']
                    if value >= 0
                    else COLOR_SCHEMES['shap_negative']
                )
                for value in top_values
            ]
            axes[2].barh(
                range(top_count),
                top_values[::-1],
                color=colors[::-1],
            )
            axes[2].set_yticks(range(top_count))
            axes[2].set_yticklabels(dim_labels[::-1], fontsize=7)
            axes[2].axvline(0, color='black', linewidth=0.8)
            axes[2].set_xlabel('SHAP value')
            axes[2].set_title(
                'Overall: top fused dimensions', fontweight='bold'
            )
        else:
            axes[2].text(
                0.5, 0.5, 'Overall SHAP unavailable',
                ha='center', va='center', transform=axes[2].transAxes
            )
            axes[2].axis('off')

        fig.suptitle(
            f'SHAP — {SID_B}\n'
            'Origins are cross-attended representations, not pure modalities',
            fontsize=12,
            fontweight='bold',
        )
        fig.tight_layout()
        shap_path = os.path.join(
            demo_dir_B, f'{SID_B}_shap_analysis.png'
        )
        fig.savefig(
            shap_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(fig)
        print(f'[XAI] Đã lưu: {shap_path}')
    else:
        print('[SKIP] Không target nào tạo được SHAP values hợp lệ.')

## 3.6 · LIME — Giải thích cục bộ (Text + Image)

> LIME đo độ nhạy quanh **mẫu hiện tại** bằng perturbation ngẫu nhiên. Kết quả không đại diện cho hành vi toàn cục của mô hình.

In [ ]:
# ── LIME: Text + Image (target=Overall Satisfaction) ─────────────────────────
TARGET_IDX_LIME = 4
lime_text_weights_B = {}
lime_text_exp_B     = None
lime_image_exp_B    = None
lime_img_paths_B    = {}

# LIME Text
print('[LIME Text] Đang tính ...')
lime_text_exp_B = run_safe(
    run_lime_text, step_name='LIME_text',
    fallback=None,
    model=model, sample=sample_B, score_index=TARGET_IDX_LIME,
    tokenizer=tokenizer, device=device,
    num_features=10, num_samples=300,
)
raw_weights_B = []
if lime_text_exp_B is not None:
    raw_weights_B = lime_text_exp_B.as_list(label=1)
    lime_text_weights_B = dict(raw_weights_B)
    print('  Top LIME words:')
    for word, w in sorted(raw_weights_B, key=lambda x: abs(x[1]), reverse=True)[:8]:
        print(f'    {("+" if w>0 else "-")} {word:<18s} {abs(w):.4f}')
    lime_factor_name = FACTOR_NAMES[TARGET_IDX_LIME]
    lt_path = f'{lime_dir_B}/{SID_B}_lime_text_{lime_factor_name}_weights.json'
    with open(lt_path, 'w', encoding='utf-8') as f:
        json.dump(raw_weights_B, f, ensure_ascii=False, indent=2)
    print(f'[XAI] Đã lưu: {lt_path}')
    lime_text_bar_B = os.path.join(
        lime_dir_B,
        f'{SID_B}_lime_text_{lime_factor_name}_bar.png',
    )
    save_lime_text_bar(
        raw_weights_B,
        lime_text_bar_B,
        f'LIME Local Text Explanation — {SID_B}',
    )
    print(f'[XAI] Đã lưu: {lime_text_bar_B}')
else:
    print('[SKIP] LIME text thất bại.')

# LIME Image
if sample_B['num_real_images'] > 0:
    print('[LIME Image] Đang tính ...')
    lime_image_exp_B = run_safe(
        run_lime_image, step_name='LIME_image',
        fallback=None,
        model=model, sample=sample_B, score_index=TARGET_IDX_LIME,
        image_processor=image_processor, device=device, num_samples=300,
    )
    if lime_image_exp_B is not None:
        lime_img_paths_B = run_safe(
            save_lime_image_explanation, step_name='save_LIME_image',
            fallback={},
            explanation=lime_image_exp_B,
            original_image=sample_B['loaded_images'][0],
            save_dir=lime_dir_B,
            sample_id=SID_B,
            target_idx=TARGET_IDX_LIME,
            factor_name=FACTOR_NAMES[TARGET_IDX_LIME],
            dpi=DEFAULT_DPI,
        ) or {}
    else:
        print('[SKIP] LIME image thất bại.')
else:
    print('[SKIP] Không có ảnh → bỏ qua LIME image.')

# 4-panel LIME visualisation
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1: text importance bar
if lime_text_weights_B:
    w_sorted = sorted(lime_text_weights_B.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
    w_names  = [w[0] for w in reversed(w_sorted)]
    w_values = [w[1] for w in reversed(w_sorted)]
    bar_cols = [COLOR_SCHEMES['shap_positive'] if v >= 0
                else COLOR_SCHEMES['shap_negative'] for v in w_values]
    axes[0].barh(range(len(w_names)), w_values, color=bar_cols)
    axes[0].set_yticks(range(len(w_names)))
    axes[0].set_yticklabels(w_names, fontsize=8)
    axes[0].axvline(0, color='black', lw=0.8)
    axes[0].set_title('LIME Text\n(từ quan trọng)', fontsize=9, fontweight='bold')
    axes[0].set_xlabel('LIME weight', fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'N/A', ha='center', va='center',
                 transform=axes[0].transAxes); axes[0].axis('off')

# Panels 2-4: image overlays
from PIL import Image as _PILImg3
for col_i, (key, title) in enumerate(
        [('positive_overlay','LIME Image (+)'),
         ('negative_overlay','LIME Image (−)'),
         ('combined_overlay','LIME Image (±)')]):
    ax = axes[col_i + 1]
    path = lime_img_paths_B.get(key)
    if path and os.path.exists(path):
        ax.imshow(np.array(_PILImg3.open(path)))
        ax.axis('off')
        ax.set_title(title, fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes); ax.axis('off')

fig.suptitle(f'LIME Explanation — {SID_B} (Overall Satisfaction)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
lime_4p = f'{demo_dir_B}/{SID_B}_lime_4panel.png'
fig.savefig(lime_4p, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[XAI] Đã lưu: {lime_4p}')


## 3.7 · AI Agent — Báo cáo tổng hợp

> GPT-4o chỉ verbalize evidence đã tạo và chỉ được gọi khi `OPENAI_API_KEY` tồn tại trong environment/Colab Secrets. Thiếu key thì section được bỏ qua an toàn.

In [ ]:
# ── Kiểm tra artifact & AI Agent ─────────────────────────────────────────────
artifact_check_B = check_sample_artifacts(SID_B, XAI_DIR)

add_cross_method_row(
    SID_B, pred_result_B,
    shap_contrib_B,
    raw_weights_B,
    case_type=CASE_B,
)

agent_output_B = None
if AGENT_AVAILABLE:
    agent_config = AgentConfig(language='vi')
    if agent_config.api_key:
        print(f'[Agent] Đang chạy AI Agent cho {SID_B} ...')
        agent_B = ExplanationAgent(agent_config)
        agent_output_B = run_safe(
            agent_B.explain_sample, step_name='AI_Agent',
            fallback=None,
            sample_id=SID_B,
            review_text=sample_B['text'],
            predictions=pred_result_B['predictions'],
            xai_dir=XAI_DIR,
            ground_truth=pred_result_B['ground_truth'],
            case_type=CASE_B,
            language='vi',
            mode='text_only',
            num_images=sample_B['num_real_images'],
            output_dir=agent_dir_B,
        )
        if agent_output_B:
            print(f"[Agent] ✅ Evidence completeness: {agent_output_B.get('evidence_completeness','N/A')}")
        else:
            print('[Agent] Không tạo được báo cáo.')
    else:
        print('[Agent] OPENAI_API_KEY không tìm thấy → bỏ qua.')
        print('  → Add OPENAI_API_KEY in Colab Secrets, then rerun this cell.')
else:
    print('[Agent] Module không khả dụng.')

print_sample_summary(SID_B, pred_result_B, artifact_check_B)
print(f'✅ HOÀN THÀNH MẪU B: {SID_B}')

---
# 🧪 PHẦN 4 — MẪU C: ĐA ẢNH PHONG PHÚ

## 4.1 · Tải mẫu & Dự đoán

In [ ]:
# ── Tải mẫu C và dự đoán ────────────────────────────────────────────────
SID_C  = SAMPLE_IDS['C']
IDX_C  = SAMPLE_INDICES['C']
CASE_C = CASE_TYPES['C']

gradcam_dir_C   = os.path.join(XAI_DIR, 'gradcam', SID_C)
attention_dir_C = os.path.join(XAI_DIR, 'attention', SID_C)
crossattn_dir_C = os.path.join(XAI_DIR, 'cross_attention', SID_C)
shap_dir_C      = os.path.join(XAI_DIR, 'shap', SID_C)
lime_dir_C      = os.path.join(XAI_DIR, 'lime', SID_C)
demo_dir_C      = os.path.join(DEMO_OUT, SID_C)
agent_dir_C     = os.path.join(AGENT_DIR, SID_C)
for output_dir in [
    gradcam_dir_C, attention_dir_C, crossattn_dir_C,
    shap_dir_C, lime_dir_C, demo_dir_C, agent_dir_C,
]:
    os.makedirs(output_dir, exist_ok=True)

print(f'Đang tải {SID_C} (idx={IDX_C}) ...')
sample_C = load_single_sample(
    csv_path=CSV_TEST,
    idx=IDX_C,
    tokenizer=tokenizer,
    image_processor=image_processor,
    image_dir=IMAGE_DIR,
    device=device,
)
print(
    f'  Text ({len(sample_C["text"])} ký tự): '
    f'{sample_C["text"][:150]} ...'
)
print(f'  Số ảnh thực: {sample_C["num_real_images"]}')
show_review_images(sample_C, SID_C)

pred_result_C = get_prediction(model, sample_C)
display_prediction_table(pred_result_C, SID_C)
plot_prediction_bars(
    pred_result_C,
    SID_C,
    save_path=os.path.join(demo_dir_C, f'{SID_C}_prediction.png'),
)

## 4.2 · Grad-CAM — Vùng ảnh quan trọng

> **Giới hạn:** image encoder dùng chung cho cả 5 đầu ra nên heatmap giữa các target có thể rất giống nhau. Demo chỉ hiển thị **Overall Satisfaction**, không dùng Grad-CAM để tuyên bố khác biệt per-target.

In [ ]:
# ── Grad-CAM: Only Overall Satisfaction (target_idx=4) ───────────────────────
# Shared encoder → cosine sim >0.95 across all 5 targets → show only overall
import datetime as _dt

TARGET_IDX_GRADCAM = 4
gradcam_results_C = {}
for img_idx in range(min(sample_C['num_real_images'], 4)):
    cam = run_safe(
        compute_gradcam_for_image, step_name=f'GradCAM img{img_idx}',
        fallback=None,
        model=model, sample=sample_C, target_idx=TARGET_IDX_GRADCAM,
        image_idx=img_idx, target_layer=target_layer, device=device,
    )
    gradcam_results_C[img_idx] = cam

n_show = min(sample_C['num_real_images'], 2)
if n_show > 0:
    import matplotlib.cm as _cm
    from PIL import Image as _PILI
    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show), squeeze=False)
    for img_idx in range(n_show):
        pil_img = sample_C['loaded_images'][img_idx]
        img224  = np.array(pil_img.convert('RGB').resize((224, 224)))
        cam     = gradcam_results_C.get(img_idx)

        axes[img_idx][0].imshow(img224)
        axes[img_idx][0].set_title(f'Ảnh {img_idx+1} — Gốc', fontsize=9)
        axes[img_idx][0].axis('off')

        if cam is not None:
            heatmap = _cm.jet(cam)[:, :, :3]
            axes[img_idx][1].imshow(heatmap)
            axes[img_idx][1].set_title('Grad-CAM Heatmap', fontsize=9)
        else:
            axes[img_idx][1].text(0.5, 0.5, 'N/A', ha='center', va='center',
                                   transform=axes[img_idx][1].transAxes)
        axes[img_idx][1].axis('off')

        if cam is not None:
            overlay = run_safe(overlay_cam_on_image, step_name='overlay',
                               cam=cam, original_image=pil_img,
                               image_size=224, colormap_name='jet', alpha=0.5)
            if overlay is not None:
                axes[img_idx][2].imshow(overlay)
                axes[img_idx][2].set_title('Overlay (CAM + Ảnh)', fontsize=9)
                cam_save = f'{gradcam_dir_C}/gradcam_img{img_idx}_overall.png'
                _PILI.fromarray(overlay).save(cam_save)
            else:
                axes[img_idx][2].axis('off')
        else:
            axes[img_idx][2].axis('off')

    fig.suptitle(f'Grad-CAM — {SID_C} (Overall Satisfaction)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    gcpath = f'{demo_dir_C}/{SID_C}_gradcam_3panel.png'
    fig.savefig(gcpath, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'[XAI] Đã lưu: {gcpath}')
else:
    print('[SKIP] Không có ảnh → bỏ qua Grad-CAM')

# metadata.json cho EvidenceLoader (chỉ ghi nếu có ít nhất 1 ảnh được tính)
if any(v is not None for v in gradcam_results_C.values()):
    gradcam_meta_C = {
        'sample_id': SID_C,
        'sample_idx': IDX_C,
        'num_images': sample_C['num_real_images'],
        'num_targets': 1,
        'target_names': ['overall'],
        'display_names': ['Overall Satisfaction'],
        'target_layer': type(target_layer).__name__,
        'device': str(device),
        'timestamp': _dt.datetime.now().isoformat(),
        'artifacts': {
            f'img{i}_overall': f'gradcam_img{i}_overall.png'
            for i, v in gradcam_results_C.items() if v is not None
        },
    }
    with open(f'{gradcam_dir_C}/metadata.json', 'w', encoding='utf-8') as f:
        json.dump(gradcam_meta_C, f, ensure_ascii=False, indent=2)


## 4.3 · PhoBERT Attention — Mức từ đã gộp

> Output hiển thị đã gộp BPE/subword thành từ đọc được. Attention mô tả luồng thông tin, không tự nó chứng minh quan hệ nhân quả.

In [ ]:
# ── PhoBERT Attention: CLS → merged word-level importance ────────────────────
attn_result_C = run_safe(
    extract_phobert_attention,
    step_name='extract_phobert_attention',
    fallback=None,
    model=model,
    input_ids=sample_C['input_ids'],
    attention_mask=sample_C['attention_mask'],
    tokenizer=tokenizer,
)

word_importances_C = []
if attn_result_C is not None:
    attentions_C = attn_result_C['attentions']
    tokens_C = attn_result_C['tokens']
    seq_len_C = attn_result_C['seq_len']
    print(
        f'  tokens={seq_len_C}, '
        f'attention shape={attentions_C.shape}'
    )

    agg_matrix_C = aggregate_attention(
        attentions_C, strategy='last_layer_mean'
    )
    cls_result_C = cls_token_importance(
        agg_matrix_C, tokens_C
    )
    word_importances_C = merge_subword_attention(
        cls_result_C['importances'],
        tokens_C,
        strategy='mean',
    )

    print(f'Top 10 từ đã gộp subword ({SID_C}):')
    for word, score in word_importances_C[:10]:
        print(f'  {word:<24s} {score:.4f}')

    word_labels = [word for word, _ in word_importances_C]
    word_values = [value for _, value in word_importances_C]
    bar_path = os.path.join(
        attention_dir_C, 'cls_importance_word_bar.png'
    )
    bar_fig = run_safe(
        plot_cls_importance_bar,
        step_name='plot_word_attention',
        fallback=None,
        tokens=word_labels,
        importances=word_values,
        title=f'PhoBERT Word-Level Attention — {SID_C}',
        save_path=None,
        top_k=15,
        dpi=DEFAULT_DPI,
    )
    if bar_fig is not None:
        bar_fig.savefig(
            bar_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(bar_fig)
        print(f'[XAI] Đã lưu: {bar_path}')

    # Raw tensors are retained for audit/reuse, but are not shown to lecturers.
    np.savez_compressed(
        os.path.join(attention_dir_C, 'raw_attention.npz'),
        attentions=attentions_C,
        tokens=np.asarray(tokens_C, dtype=object),
    )
    word_payload = {
        'word_importances': [
            {'word': word, 'importance': float(score)}
            for word, score in word_importances_C
        ]
    }
    with open(
        os.path.join(attention_dir_C, 'word_importance.json'),
        'w',
        encoding='utf-8',
    ) as file:
        json.dump(word_payload, file, ensure_ascii=False, indent=2)
    with open(
        os.path.join(attention_dir_C, 'topk_tokens.json'),
        'w',
        encoding='utf-8',
    ) as file:
        json.dump(
            [
                {'token': word, 'importance': float(score)}
                for word, score in word_importances_C[:15]
            ],
            file,
            ensure_ascii=False,
            indent=2,
        )
    print(
        '[Attention] Visible output uses merged words; '
        'raw BPE/subword fragments are stored only as machine-readable data.'
    )
else:
    print('[SKIP] Không trích xuất được PhoBERT attention.')

## 4.4 · Cross-Attention — Tương tác hai chiều

- **Token → Patch:** khi xử lý một từ, mô hình phân bổ chú ý lên các patch ảnh nào?
- **Patch → Token:** từ một patch ảnh, mô hình liên kết ngược tới các từ nào?

Hai hướng dùng phép chuẩn hóa khác nhau; không diễn giải chúng như hai ma trận chuyển vị.

In [ ]:
# ── Bidirectional Cross-Attention: Token → Patch and Patch → Token ────────────
cross_result_C = run_safe(
    extract_cross_attention,
    step_name='extract_cross_attention',
    fallback=None,
    model=model,
    sample=sample_C,
    tokenizer=tokenizer,
)

t2i_C = None
i2t_C = None
if cross_result_C is not None:
    t2i_C = cross_result_C['t2i_attn']
    i2t_C = cross_result_C['i2t_attn']
    raw_tokens_C = cross_result_C['tokens']
    word_merge_C = run_safe(
        merge_cross_attention_words,
        step_name='merge_cross_attention_words',
        fallback=([], None, None),
        tokens=raw_tokens_C,
        t2i_attn=t2i_C,
        i2t_attn=i2t_C,
    )
    ca_words_C, t2i_words_C, i2t_words_C = word_merge_C

    if t2i_words_C is not None and i2t_words_C is not None:
        num_words, num_patches = t2i_words_C.shape
        grid_h = int(np.sqrt(num_patches))
        grid_w = (
            grid_h if grid_h * grid_h == num_patches
            else num_patches
        )
        patch_labels = (
            [
                f'({row},{col})'
                for row in range(grid_h)
                for col in range(grid_h)
            ]
            if grid_h * grid_h == num_patches
            else [str(index) for index in range(num_patches)]
        )

        word_strength = t2i_words_C.max(axis=1)
        top_word_indices = np.argsort(word_strength)[::-1][
            :min(20, num_words)
        ]
        patch_strength = i2t_words_C.max(axis=1)
        top_patch_indices = np.argsort(patch_strength)[::-1][
            :min(20, num_patches)
        ]

        visible_words = [ca_words_C[i] for i in top_word_indices]
        visible_patches = [patch_labels[i] for i in top_patch_indices]
        visible_t2i = t2i_words_C[top_word_indices, :]
        visible_i2t = i2t_words_C[
            np.ix_(top_patch_indices, top_word_indices)
        ]

        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        try:
            import seaborn as sns
            sns.heatmap(
                visible_t2i,
                xticklabels=patch_labels,
                yticklabels=visible_words,
                cmap='viridis',
                ax=axes[0],
                cbar_kws={'shrink': 0.65},
            )
            sns.heatmap(
                visible_i2t,
                xticklabels=visible_words,
                yticklabels=visible_patches,
                cmap='magma',
                ax=axes[1],
                cbar_kws={'shrink': 0.65},
            )
        except ImportError:
            axes[0].imshow(visible_t2i, aspect='auto', cmap='viridis')
            axes[1].imshow(visible_i2t, aspect='auto', cmap='magma')

        axes[0].set_title(
            'Token → Patch\nKhi đọc từ này, mô hình nhìn vùng ảnh nào?',
            fontsize=10,
            fontweight='bold',
        )
        axes[0].set_xlabel('Image patches')
        axes[0].set_ylabel('Merged words')
        axes[0].tick_params(axis='both', labelsize=6)

        axes[1].set_title(
            'Patch → Token\nKhi nhìn patch này, mô hình liên kết từ nào?',
            fontsize=10,
            fontweight='bold',
        )
        axes[1].set_xlabel('Merged words')
        axes[1].set_ylabel('Top image patches')
        axes[1].tick_params(axis='both', labelsize=7)

        fig.suptitle(
            f'Bidirectional Cross-Attention — {SID_C}',
            fontsize=13,
            fontweight='bold',
        )
        fig.tight_layout()
        cross_path = os.path.join(
            demo_dir_C, f'{SID_C}_cross_attention.png'
        )
        fig.savefig(
            cross_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(fig)
        print(f'[XAI] Đã lưu: {cross_path}')
        print(
            '[Giải thích hai chiều] Token→Patch cho biết vùng ảnh được truy vấn '
            'khi xử lý một từ; Patch→Token cho biết từ nào được một vùng ảnh '
            'liên kết. Hai ma trận dùng softmax theo hai trục khác nhau nên '
            'không được diễn giải như phép chuyển vị của nhau.'
        )

        np.savez_compressed(
            os.path.join(
                crossattn_dir_C, 'cross_attention_raw.npz'
            ),
            t2i=t2i_C,
            i2t=i2t_C,
            word_t2i=t2i_words_C,
            word_i2t=i2t_words_C,
            words=np.asarray(ca_words_C, dtype=object),
        )

        top5_indices = np.argsort(word_strength)[::-1][:5]
        entropy = -np.sum(
            t2i_words_C * np.log2(t2i_words_C + 1e-12),
            axis=1,
        )
        summary = {
            'sample_id': SID_C,
            'mean_token_entropy': float(entropy.mean()),
            'top_5_tokens': [
                {
                    'token': ca_words_C[i],
                    'importance': float(word_strength[i]),
                }
                for i in top5_indices
            ],
            'interpretation': {
                'token_to_patch': (
                    'For each merged word, distribution over image patches.'
                ),
                'patch_to_token': (
                    'For each image patch, distribution over merged words.'
                ),
            },
        }
        with open(
            os.path.join(
                crossattn_dir_C, 'cross_attention_summary.json'
            ),
            'w',
            encoding='utf-8',
        ) as file:
            json.dump(summary, file, ensure_ascii=False, indent=2)

        top_pairs = []
        for word_idx in top5_indices:
            patch_idx = int(np.argmax(t2i_words_C[word_idx]))
            top_pairs.append({
                'token': ca_words_C[word_idx],
                'patch_row': (
                    patch_idx // grid_h if grid_h * grid_h == num_patches
                    else 0
                ),
                'patch_col': (
                    patch_idx % grid_h if grid_h * grid_h == num_patches
                    else patch_idx
                ),
                'attention': float(
                    t2i_words_C[word_idx, patch_idx]
                ),
            })
        with open(
            os.path.join(crossattn_dir_C, 'token_patch_topk.json'),
            'w',
            encoding='utf-8',
        ) as file:
            json.dump(top_pairs, file, ensure_ascii=False, indent=2)
    else:
        print('[SKIP] Không thể gộp Cross-Attention ở mức từ.')
else:
    print('[SKIP] Không trích xuất được Cross-Attention.')

## 4.5 · SHAP — Đóng góp fused embedding [1024]

> `text-origin` (dims 0:512) và `image-origin` (dims 512:1024) đều là biểu diễn **sau Cross-Attention**. Đây không phải hai modality thuần túy. SHAP dùng baseline nhiều mẫu, không dùng chính sample làm baseline.

In [ ]:
# ── SHAP on fused embedding [1024], using a multi-sample baseline ─────────────
fused_result_C = run_safe(
    extract_fused_embeddings,
    step_name='extract_fused_embeddings',
    fallback=(None, None, None),
    model=model,
    dataloader=[sample_to_batch(sample_C)],
    device=device,
    max_samples=1,
)
fused_C = fused_result_C[0] if fused_result_C else None

shap_values_by_factor_C = {}
shap_contrib_by_factor_C = {}
shap_vals_C = None
shap_contrib_C = None

if fused_C is None:
    print('[SKIP] Không extract được fused embedding.')
elif SHAP_BACKGROUND is None:
    print(
        '[SKIP] SHAP không có multi-sample background hợp lệ; '
        'không dùng chính sample làm baseline vì sẽ tạo attribution bằng 0.'
    )
else:
    print(
        f'[SHAP] Background={tuple(SHAP_BACKGROUND.shape)}, '
        f'sample={tuple(fused_C.shape)}'
    )
    for score_index, factor_name in enumerate(FACTOR_NAMES):
        wrapper = FusionHeadWrapper(model.head, score_index=score_index)
        shap_result = run_safe(
            compute_shap_values,
            step_name=f'SHAP_{factor_name}',
            fallback=(None, None),
            wrapper=wrapper,
            background=SHAP_BACKGROUND,
            samples=fused_C,
        )
        values = shap_result[0] if shap_result else None
        if values is None or len(values) == 0:
            continue
        sample_values = np.asarray(values[0], dtype=float).reshape(-1)
        if sample_values.size != FUSED_DIM or not np.isfinite(
            sample_values
        ).all():
            print(
                f'[SKIP] SHAP {factor_name}: invalid shape/value '
                f'{sample_values.shape}'
            )
            continue
        contribution = modality_contribution(sample_values)
        shap_values_by_factor_C[factor_name] = sample_values
        shap_contrib_by_factor_C[factor_name] = contribution
        print(
            f'  {DISPLAY_NAMES[score_index]:<22s} '
            f'text-origin={contribution["text_pct"]:5.1f}% | '
            f'image-origin={contribution["image_pct"]:5.1f}%'
        )

    shap_vals_C = shap_values_by_factor_C.get('overall')
    shap_contrib_C = shap_contrib_by_factor_C.get('overall')

    if shap_contrib_by_factor_C:
        contribution_path = os.path.join(
            shap_dir_C, 'shap_modality_contribution.json'
        )
        with open(
            contribution_path, 'w', encoding='utf-8'
        ) as file:
            json.dump(
                shap_contrib_by_factor_C,
                file,
                ensure_ascii=False,
                indent=2,
            )
        np.savez_compressed(
            os.path.join(shap_dir_C, 'raw_shap_values.npz'),
            **shap_values_by_factor_C,
        )
        print(f'[XAI] Đã lưu: {contribution_path}')

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        if (
            shap_contrib_C is not None
            and shap_contrib_C['text_abs']
            + shap_contrib_C['image_abs'] > 1e-12
        ):
            axes[0].pie(
                [
                    shap_contrib_C['text_abs'],
                    shap_contrib_C['image_abs'],
                ],
                labels=[
                    f'Text-origin\n{shap_contrib_C["text_pct"]:.1f}%',
                    f'Image-origin\n{shap_contrib_C["image_pct"]:.1f}%',
                ],
                colors=[
                    COLOR_SCHEMES['modality_colors']['text'],
                    COLOR_SCHEMES['modality_colors']['image'],
                ],
                autopct='%1.1f%%',
                startangle=90,
            )
            axes[0].set_title(
                'Overall Satisfaction\nModality contribution',
                fontweight='bold',
            )
        else:
            axes[0].text(
                0.5, 0.5, 'Overall SHAP unavailable',
                ha='center', va='center', transform=axes[0].transAxes
            )
            axes[0].axis('off')

        available_factors = [
            factor for factor in FACTOR_NAMES
            if factor in shap_contrib_by_factor_C
        ]
        x_positions = np.arange(len(available_factors))
        text_pct = [
            shap_contrib_by_factor_C[factor]['text_pct']
            for factor in available_factors
        ]
        image_pct = [
            shap_contrib_by_factor_C[factor]['image_pct']
            for factor in available_factors
        ]
        axes[1].bar(
            x_positions,
            text_pct,
            label='Text-origin',
            color=COLOR_SCHEMES['modality_colors']['text'],
        )
        axes[1].bar(
            x_positions,
            image_pct,
            bottom=text_pct,
            label='Image-origin',
            color=COLOR_SCHEMES['modality_colors']['image'],
        )
        axes[1].set_xticks(x_positions)
        axes[1].set_xticklabels(
            [
                DISPLAY_NAMES[FACTOR_NAMES.index(factor)]
                for factor in available_factors
            ],
            rotation=25,
            ha='right',
            fontsize=8,
        )
        axes[1].set_ylim(0, 100)
        axes[1].set_ylabel('|SHAP| contribution (%)')
        axes[1].set_title(
            'Per-target origin contribution', fontweight='bold'
        )
        axes[1].legend(fontsize=8)

        if shap_vals_C is not None:
            top_count = min(20, shap_vals_C.size)
            top_indices = np.argsort(np.abs(shap_vals_C))[
                -top_count:
            ][::-1]
            top_values = shap_vals_C[top_indices]
            dim_labels = [
                (
                    f'T{index}'
                    if index < CROSS_ATTN_HIDDEN_DIM
                    else f'I{index - CROSS_ATTN_HIDDEN_DIM}'
                )
                for index in top_indices
            ]
            colors = [
                (
                    COLOR_SCHEMES['shap_positive']
                    if value >= 0
                    else COLOR_SCHEMES['shap_negative']
                )
                for value in top_values
            ]
            axes[2].barh(
                range(top_count),
                top_values[::-1],
                color=colors[::-1],
            )
            axes[2].set_yticks(range(top_count))
            axes[2].set_yticklabels(dim_labels[::-1], fontsize=7)
            axes[2].axvline(0, color='black', linewidth=0.8)
            axes[2].set_xlabel('SHAP value')
            axes[2].set_title(
                'Overall: top fused dimensions', fontweight='bold'
            )
        else:
            axes[2].text(
                0.5, 0.5, 'Overall SHAP unavailable',
                ha='center', va='center', transform=axes[2].transAxes
            )
            axes[2].axis('off')

        fig.suptitle(
            f'SHAP — {SID_C}\n'
            'Origins are cross-attended representations, not pure modalities',
            fontsize=12,
            fontweight='bold',
        )
        fig.tight_layout()
        shap_path = os.path.join(
            demo_dir_C, f'{SID_C}_shap_analysis.png'
        )
        fig.savefig(
            shap_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(fig)
        print(f'[XAI] Đã lưu: {shap_path}')
    else:
        print('[SKIP] Không target nào tạo được SHAP values hợp lệ.')

## 4.6 · LIME — Giải thích cục bộ (Text + Image)

> LIME đo độ nhạy quanh **mẫu hiện tại** bằng perturbation ngẫu nhiên. Kết quả không đại diện cho hành vi toàn cục của mô hình.

In [ ]:
# ── LIME: Text + Image (target=Overall Satisfaction) ─────────────────────────
TARGET_IDX_LIME = 4
lime_text_weights_C = {}
lime_text_exp_C     = None
lime_image_exp_C    = None
lime_img_paths_C    = {}

# LIME Text
print('[LIME Text] Đang tính ...')
lime_text_exp_C = run_safe(
    run_lime_text, step_name='LIME_text',
    fallback=None,
    model=model, sample=sample_C, score_index=TARGET_IDX_LIME,
    tokenizer=tokenizer, device=device,
    num_features=10, num_samples=300,
)
raw_weights_C = []
if lime_text_exp_C is not None:
    raw_weights_C = lime_text_exp_C.as_list(label=1)
    lime_text_weights_C = dict(raw_weights_C)
    print('  Top LIME words:')
    for word, w in sorted(raw_weights_C, key=lambda x: abs(x[1]), reverse=True)[:8]:
        print(f'    {("+" if w>0 else "-")} {word:<18s} {abs(w):.4f}')
    lime_factor_name = FACTOR_NAMES[TARGET_IDX_LIME]
    lt_path = f'{lime_dir_C}/{SID_C}_lime_text_{lime_factor_name}_weights.json'
    with open(lt_path, 'w', encoding='utf-8') as f:
        json.dump(raw_weights_C, f, ensure_ascii=False, indent=2)
    print(f'[XAI] Đã lưu: {lt_path}')
    lime_text_bar_C = os.path.join(
        lime_dir_C,
        f'{SID_C}_lime_text_{lime_factor_name}_bar.png',
    )
    save_lime_text_bar(
        raw_weights_C,
        lime_text_bar_C,
        f'LIME Local Text Explanation — {SID_C}',
    )
    print(f'[XAI] Đã lưu: {lime_text_bar_C}')
else:
    print('[SKIP] LIME text thất bại.')

# LIME Image
if sample_C['num_real_images'] > 0:
    print('[LIME Image] Đang tính ...')
    lime_image_exp_C = run_safe(
        run_lime_image, step_name='LIME_image',
        fallback=None,
        model=model, sample=sample_C, score_index=TARGET_IDX_LIME,
        image_processor=image_processor, device=device, num_samples=300,
    )
    if lime_image_exp_C is not None:
        lime_img_paths_C = run_safe(
            save_lime_image_explanation, step_name='save_LIME_image',
            fallback={},
            explanation=lime_image_exp_C,
            original_image=sample_C['loaded_images'][0],
            save_dir=lime_dir_C,
            sample_id=SID_C,
            target_idx=TARGET_IDX_LIME,
            factor_name=FACTOR_NAMES[TARGET_IDX_LIME],
            dpi=DEFAULT_DPI,
        ) or {}
    else:
        print('[SKIP] LIME image thất bại.')
else:
    print('[SKIP] Không có ảnh → bỏ qua LIME image.')

# 4-panel LIME visualisation
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1: text importance bar
if lime_text_weights_C:
    w_sorted = sorted(lime_text_weights_C.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
    w_names  = [w[0] for w in reversed(w_sorted)]
    w_values = [w[1] for w in reversed(w_sorted)]
    bar_cols = [COLOR_SCHEMES['shap_positive'] if v >= 0
                else COLOR_SCHEMES['shap_negative'] for v in w_values]
    axes[0].barh(range(len(w_names)), w_values, color=bar_cols)
    axes[0].set_yticks(range(len(w_names)))
    axes[0].set_yticklabels(w_names, fontsize=8)
    axes[0].axvline(0, color='black', lw=0.8)
    axes[0].set_title('LIME Text\n(từ quan trọng)', fontsize=9, fontweight='bold')
    axes[0].set_xlabel('LIME weight', fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'N/A', ha='center', va='center',
                 transform=axes[0].transAxes); axes[0].axis('off')

# Panels 2-4: image overlays
from PIL import Image as _PILImg3
for col_i, (key, title) in enumerate(
        [('positive_overlay','LIME Image (+)'),
         ('negative_overlay','LIME Image (−)'),
         ('combined_overlay','LIME Image (±)')]):
    ax = axes[col_i + 1]
    path = lime_img_paths_C.get(key)
    if path and os.path.exists(path):
        ax.imshow(np.array(_PILImg3.open(path)))
        ax.axis('off')
        ax.set_title(title, fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes); ax.axis('off')

fig.suptitle(f'LIME Explanation — {SID_C} (Overall Satisfaction)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
lime_4p = f'{demo_dir_C}/{SID_C}_lime_4panel.png'
fig.savefig(lime_4p, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[XAI] Đã lưu: {lime_4p}')


## 4.7 · AI Agent — Báo cáo tổng hợp

> GPT-4o chỉ verbalize evidence đã tạo và chỉ được gọi khi `OPENAI_API_KEY` tồn tại trong environment/Colab Secrets. Thiếu key thì section được bỏ qua an toàn.

In [ ]:
# ── Kiểm tra artifact & AI Agent ─────────────────────────────────────────────
artifact_check_C = check_sample_artifacts(SID_C, XAI_DIR)

add_cross_method_row(
    SID_C, pred_result_C,
    shap_contrib_C,
    raw_weights_C,
    case_type=CASE_C,
)

agent_output_C = None
if AGENT_AVAILABLE:
    agent_config = AgentConfig(language='vi')
    if agent_config.api_key:
        print(f'[Agent] Đang chạy AI Agent cho {SID_C} ...')
        agent_C = ExplanationAgent(agent_config)
        agent_output_C = run_safe(
            agent_C.explain_sample, step_name='AI_Agent',
            fallback=None,
            sample_id=SID_C,
            review_text=sample_C['text'],
            predictions=pred_result_C['predictions'],
            xai_dir=XAI_DIR,
            ground_truth=pred_result_C['ground_truth'],
            case_type=CASE_C,
            language='vi',
            mode='text_only',
            num_images=sample_C['num_real_images'],
            output_dir=agent_dir_C,
        )
        if agent_output_C:
            print(f"[Agent] ✅ Evidence completeness: {agent_output_C.get('evidence_completeness','N/A')}")
        else:
            print('[Agent] Không tạo được báo cáo.')
    else:
        print('[Agent] OPENAI_API_KEY không tìm thấy → bỏ qua.')
        print('  → Add OPENAI_API_KEY in Colab Secrets, then rerun this cell.')
else:
    print('[Agent] Module không khả dụng.')

print_sample_summary(SID_C, pred_result_C, artifact_check_C)
print(f'✅ HOÀN THÀNH MẪU C: {SID_C}')

---
# 📊 PHẦN 5 — SO SÁNH CROSS-SAMPLE

Tổng hợp kết quả 3 mẫu: MAE per target, đóng góp modality SHAP, top LIME words.

In [ ]:
if CROSS_METHOD_ROWS:
    df_cross = pd.DataFrame(CROSS_METHOD_ROWS)
    print('╔══ BẢNG SO SÁNH 3 MẪU ══════════════════════════════════╗')
    print(df_cross.to_string(index=False))
    print('╚═════════════════════════════════════════════════════════╝')
    display(df_cross)
else:
    print('[WARN] Chưa có dữ liệu cross-method.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

all_samples = [
    ('A', SAMPLE_IDS['A'], pred_result_A, shap_contrib_A),
    ('B', SAMPLE_IDS['B'], pred_result_B, shap_contrib_B),
    ('C', SAMPLE_IDS['C'], pred_result_C, shap_contrib_C),
]

# Panel 1: MAE per target
x = range(len(TARGET_NAMES))
width = 0.25
for i, (letter, sid, pred_res, _) in enumerate(all_samples):
    ae_vals = [pred_res['absolute_errors'][n] for n in TARGET_NAMES]
    offset  = (i - 1) * width
    axes[0].bar([xi + offset for xi in x], ae_vals, width,
                label=f'Mẫu {letter}', alpha=0.8)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(['Food','Price','Atmos','Service','Overall'],
                         fontsize=8, rotation=20, ha='right')
axes[0].set_ylabel('Absolute Error', fontsize=9)
axes[0].set_title('MAE per Target\n(3 mẫu)', fontsize=10, fontweight='bold')
axes[0].legend(fontsize=7)
axes[0].axhline(y=0.5, color='green', linestyle='--', alpha=0.5)

# Panel 2: SHAP modality contribution stacked bar
shap_t = [c.get('text_pct', 0) if c else 0 for _, _, _, c in all_samples]
shap_i = [c.get('image_pct', 0) if c else 0 for _, _, _, c in all_samples]
x2 = range(len(all_samples))
labels2 = [f'Mẫu {l}' for l, _, _, _ in all_samples]
axes[1].bar(x2, shap_t, label='Text-origin',
            color=COLOR_SCHEMES['modality_colors']['text'], alpha=0.8)
axes[1].bar(x2, shap_i, bottom=shap_t, label='Image-origin',
            color=COLOR_SCHEMES['modality_colors']['image'], alpha=0.8)
axes[1].set_xticks(list(x2))
axes[1].set_xticklabels(labels2, fontsize=9)
axes[1].set_ylabel('SHAP Contribution (%)', fontsize=9)
axes[1].set_title('SHAP Modality\nContribution', fontsize=10, fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 100)

# Panel 3: Overall MAE
maes = [r['mean_mae'] for _, _, r, _ in all_samples]
cols_mae = ['#43A047', '#E53935', '#1E88E5']
bars3 = axes[2].bar(
    ['Mẫu A\n(Chính xác)', 'Mẫu B\n(Lỗi)', 'Mẫu C\n(Đa ảnh)'],
    maes, color=cols_mae, alpha=0.85, edgecolor='white')
for bar, mae_v in zip(bars3, maes):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{mae_v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[2].set_ylabel('Mean MAE', fontsize=9)
axes[2].set_title('Overall MAE\n(3 mẫu)', fontsize=10, fontweight='bold')
axes[2].axhline(y=0.5, color='green', linestyle='--', alpha=0.6)

fig.suptitle('Cross-Sample Comparison — XAI Pipeline',
             fontsize=13, fontweight='bold')
plt.tight_layout()
cross_path = f'{DEMO_OUT}/cross_sample_comparison.png'
fig.savefig(cross_path, dpi=THESIS_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'[XAI] Đã lưu: {cross_path}')


In [ ]:
# ── Tổng kết demo ────────────────────────────────────────────────────────────
import datetime as _dt
import glob as _glob

print('╔══════════════════════════════════════════════════════════╗')
print('║            TỔNG KẾT DEMO XAI + AI AGENT                 ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Pipeline đầy đủ:                                        ║')
print('║    1. ✅ Dự đoán (CrossAttentionFusion)                  ║')
print('║    2. ✅ Grad-CAM (vùng ảnh quan trọng)                  ║')
print('║    3. ✅ PhoBERT Attention (từ nổi bật)                   ║')
print('║    4. ✅ Cross-Attention T2I + I2T                        ║')
print('║    5. ✅ SHAP (text/image modality contribution)          ║')
print('║    6. ✅ LIME (text + image, 4-panel)                     ║')
agent_completed = any(
    output is not None
    for output in [agent_output_A, agent_output_B, agent_output_C]
)
agent_status = (
    '✅ AI Agent (GPT-4o) đã tạo ít nhất một báo cáo'
    if agent_completed
    else '⏭️ AI Agent đã skip (thiếu key hoặc lỗi optional)'
)
print(f'║    7. {agent_status:<50s} ║')
print('║                                                          ║')
print('║  3 mẫu: Chính xác | Lỗi/Xung đột | Đa ảnh phong phú   ║')
print('╚══════════════════════════════════════════════════════════╝')

xai_count  = len(_glob.glob(f'{XAI_DIR}/**/*', recursive=True))
demo_count = len(_glob.glob(f'{DEMO_OUT}/**/*', recursive=True))
print(f'\nTổng files XAI artifacts (Drive, {XAI_DIR}): {xai_count}')
print(f'Tổng files demo presentation (Drive, {DEMO_OUT}): {demo_count}')

# ── Manifest tổng kết demo (lưu vào Drive) ───────────────────────────────────
demo_manifest = {
    'experiment_id': EXP_ID,
    'timestamp': _dt.datetime.now().isoformat(),
    'samples': {
        letter: {
            'sample_id': sid,
            'case_type': CASE_TYPES[letter],
            'mean_mae': pred_res['mean_mae'],
            'completeness': artifact_check.get('completeness', 0),
        }
        for letter, sid, pred_res, artifact_check in [
            ('A', SID_A, pred_result_A, artifact_check_A),
            ('B', SID_B, pred_result_B, artifact_check_B),
            ('C', SID_C, pred_result_C, artifact_check_C),
        ]
    },
    'xai_dir': XAI_DIR,
    'demo_out': DEMO_OUT,
}
manifest_path = f'{DEMO_OUT}/demo_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(demo_manifest, f, ensure_ascii=False, indent=2)
print(f'\n✅ Manifest đã lưu: {manifest_path}')
